#Configuration



In [ ]:
import os
import pandas as pd
import requests
import json
import base64
import time
import glob

# DataForSEO Credentials
login = os.environ["DATAFORSEO_LOGIN"]
password = os.environ["DATAFORSEO_PASSWORD"]

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

base_path = "/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords"
if not os.path.exists(base_path):
    os.makedirs(base_path)

# Prepare Keyword CSV File

In [ ]:
# Define the input directory path
input_dir = os.path.join(base_path, "input")
if not os.path.exists(input_dir):
    os.makedirs(input_dir)
    print(f"Created directory: {input_dir}")


sample_data = {
    'location_name': ['Malone_NY_S', 'SaranacLake_NY_S', 'Syracuse_NY_M','Buffalo_NY_L','NYC_NY_L',
                      'Eureka_CA_S', 'FortBragg_CA_S', 'Modesto_CA_M', 'SanFrancisco_CA_L','LA_CA_L',
                      'Vidalia_GA_S', 'Toccoa_GA_S', 'Macon_GA_M', 'Atlanta_GA_L', 'Augusta_GA_L',
                      ],
    'location_code': [1026588, 1023342, 1027001, 1022764, 1023191,
                      1013774, 1013806, 1014019, 1014221, 1013962,
                      1015545, 1015533, 1015427, 1015254, 1015256],
    'keywords_General_Dentist': [
        'dental clinic',
        'family dentist',
        'dentist',
        'general dentistry',
        'cosmetic dentist',
        None,
        None,
        None,
        None,
        None,
        None,
        None,
        None,
        None,
        None
    ],
    'keywords_Special_Dentist': [
        'orthodontist',
        'pediatric dentist',
        'periodontist',
        'prosthodontist',
        None,
        None,
        None,
        None,
        None,
        None,
        None,
        None,
        None,
        None,
        None
    ],
    'keywords_Surgery_Dentist': [
        'oral surgeon',
        'dental implants',
        'emergency dentist',
        None,
        None,
        None,
        None,
        None,
        None,
        None,
        None,
        None,
        None,
        None,
        None
    ]
}
sample_df = pd.DataFrame(sample_data)
csv_path = os.path.join(input_dir, "keyword_dentist.csv") # Save to input directory
sample_df.to_csv(csv_path, index=False)

print(f"Sample file saved to: {csv_path}")

from IPython.display import display
display(sample_df)

# POST Tasks to DataForSEO
## Wait 20min after running this cell

In [ ]:
def post_dataforseo_task(post_url, location_code, keyword, depth=100):
    post_payload = [{
        "location_code": location_code,
        "language_code": "en",
        "keyword": keyword,
        "depth": depth
    }]

    cred_string = f"{login}:{password}"
    cred_base64 = base64.b64encode(cred_string.encode("utf-8")).decode("utf-8")
    headers = {
        'Authorization': f'Basic {cred_base64}',
        'Content-Type': 'application/json'
    }

    print(f"  POST task: Location Code={location_code}, Keyword='{keyword}'")

    try:
        response = requests.post(post_url, headers=headers, json=post_payload)
        response.raise_for_status() # Check for HTTP status code errors
        result = response.json()

        if result and result.get("tasks") and result["tasks"][0].get("id"):
            task_id = result["tasks"][0]["id"]
            print(f"  Task POST successfully for keyword: '{keyword}'. Task ID: {task_id}")
            return task_id
        else:
            print(f"  Task POST successful, but no Task ID found in response. Keyword: '{keyword}'. Response: {result}")
            return None

    except requests.exceptions.RequestException as e:
        print(f"  Error POST task for keyword: '{keyword}': {e}")
        return None
    except json.JSONDecodeError:
        print(f"  Task POST successful, but could not parse JSON response. Keyword: '{keyword}'. Response text: {response.text}")
        return None




In [ ]:
print("--- Starting Task POST ---")
input_dir = os.path.join(base_path, "input")
temp_dir = os.path.join(base_path, "temp")
output_dir = os.path.join(base_path, "output")

if not os.path.exists(input_dir):
    os.makedirs(input_dir)
    print(f"Created directory: {input_dir}")
if not os.path.exists(temp_dir):
    os.makedirs(temp_dir)
    print(f"Created directory: {temp_dir}")
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"Created directory: {output_dir}")


csv_path = os.path.join(input_dir, "keyword_dentist.csv")
try:
    df = pd.read_csv(csv_path)
except FileNotFoundError:
    print(f"Error: '{csv_path}' not found. Please run the previous cell first.")
    tasks_to_submit_df = pd.DataFrame()
else:
    tasks_to_submit_df = df.copy()

task_list_csv_path = os.path.join(temp_dir, "task_list.csv")

# Load already posted tasks to avoid re-posting
existing_tasks = set()
try:
    existing_tasks_df = pd.read_csv(task_list_csv_path)
    # Create a set of (location_name, keyword, api_type) tuples
    for index, row in existing_tasks_df.iterrows():
        existing_tasks.add((row['location_name'], row['keyword'], row['api_type']))
    print(f"Loaded {len(existing_tasks)} existing tasks to skip.")
except FileNotFoundError:
    print("No existing task file found. Will post all tasks.")


write_header = not os.path.exists(task_list_csv_path)

api_endpoints = {
    "local_finder": "https://api.dataforseo.com/v3/serp/google/local_finder/task_post",
    "maps": "https://api.dataforseo.com/v3/serp/google/maps/task_post"
}

if not tasks_to_submit_df.empty:
    keyword_columns = [col for col in tasks_to_submit_df.columns if col.startswith('keyword')]

    if not keyword_columns:
        print("Error: No keyword columns found.")
    else:
        for keyword_col in keyword_columns:
            print(f"Processing keyword column: '{keyword_col}'")
            keywords_in_column = tasks_to_submit_df[keyword_col].dropna().tolist()

            if not keywords_in_column:
                print(f"  Keyword column '{keyword_col}' has no keywords. Skipping.")
                continue

            from itertools import combinations
            keyword_combinations = []
            # for i in range(1, min(len(keywords_in_column), 3) + 1):
            for i in range(1, len(keywords_in_column) + 1):
                 keyword_combinations.extend(list(combinations(keywords_in_column, i)))

            if not keyword_combinations:
                 print(f"  Could not generate combinations for keyword column '{keyword_col}'. Skipping.")
                 continue

            print(f"  Keyword combinations for column '{keyword_col}': {keyword_combinations}")

            for location_index, location_row in tasks_to_submit_df.iterrows():
                location_name = location_row['location_name']
                location_code_val = location_row['location_code']

                if pd.isna(location_code_val):
                    print(f"  Skipping task POST for location '{location_name}' due to missing location_code.")
                    continue

                location_code = int(location_code_val)
                print(f"  Processing tasks for location '{location_name}' ({location_code}):")

                for combo in keyword_combinations:
                    combined_keyword = "+".join(combo)

                    for api_name, api_url in api_endpoints.items():
                        # Skip posted task
                        if (location_name, combined_keyword, api_name) in existing_tasks:
                            print(f"    Skipping already posted task: Keyword='{combined_keyword}', API='{api_name}'")
                            continue # Move to the next api_name

                        print(f"    Combined keyword: '{combined_keyword}'")
                        raw_output_dir = os.path.join(temp_dir, api_name)
                        if not os.path.exists(raw_output_dir):
                            os.makedirs(raw_output_dir)
                            print(f"Created directory: {raw_output_dir}")

                        task_id = post_dataforseo_task(api_url, location_code, combined_keyword)
                        if task_id:
                            safe_combined_keyword = combined_keyword.replace(' ', '_').replace('/', '_').replace('\\\\', '_')
                            raw_json_filename = f"{location_name}_{safe_combined_keyword}_{api_name}.json"
                            raw_json_path = os.path.join(raw_output_dir, raw_json_filename)

                            current_task_df = pd.DataFrame([{
                                "task_id": task_id,
                                "api_type": api_name,
                                "location_name": location_name,
                                "keyword": combined_keyword,
                                "raw_json_path": raw_json_path
                            }])

                            current_task_df.to_csv(task_list_csv_path, mode='a', header=write_header, index=False)
                            write_header = False
                            print(f"  Task info appended to: '{task_list_csv_path}'")

                        time.sleep(1)


    print(f"All new tasks posted and incrementally saved to '{task_list_csv_path}').")
    print("!!! IMPORTANT: Please wait 20 minutes before running the next cell to allow results to become available. !!!")
else:
    print("Keyword database file not found or empty.")

#Get Task Results
## Wait 20 mins after running previous cell

In [ ]:
def get_dataforseo_results(task_id, api_type, raw_json_path):
    if api_type == "local_finder":
        get_url_template = "https://api.dataforseo.com/v3/serp/google/local_finder/task_get/advanced/{}"
    elif api_type == "maps":
        get_url_template = "https://api.dataforseo.com/v3/serp/google/maps/task_get/advanced/{}"
    else:
        print(f"  Unknown API type: {api_type}")
        return

    get_url = get_url_template.format(task_id)

    cred_string = f"{login}:{password}"
    cred_base64 = base64.b64encode(cred_string.encode("utf-8")).decode("utf-8")
    headers = {'Authorization': f'Basic {cred_base64}'}

    try:
        response = requests.get(get_url, headers=headers)
        response.raise_for_status()
        result_data = response.json()

        raw_data_dir = os.path.dirname(raw_json_path)
        if not os.path.exists(raw_data_dir):
            os.makedirs(raw_data_dir)
            print(f"Created directory: {raw_data_dir}")

        with open(raw_json_path, "w", encoding="utf-8") as f:
            json.dump(result_data, f, indent=4, ensure_ascii=False)
        print(f"  Saved JSON results for Task ID {task_id} to {os.path.basename(raw_json_path)}") # Updated message
    except requests.exceptions.RequestException as e:
        print(f"  Error getting results for Task ID {task_id}: {e}")

In [ ]:
print("--- Get Task Results ---")
temp_dir = os.path.join(base_path, "temp")
task_list_path = os.path.join(temp_dir, "task_list.csv")
tasks_to_get = []
try:
    # Read the task list from CSV
    task_list_df = pd.read_csv(task_list_path)
    # Convert df rows to a list of dictionaries
    tasks_to_get = task_list_df.to_dict('records')
except FileNotFoundError:
    print(f"Error: '{task_list_path}' not found. Please run the task submission cell first.")

if tasks_to_get:
    print(f"Found {len(tasks_to_get)} tasks to get results for.")
    for task in tasks_to_get:

        # Skip result file already exists
        if os.path.exists(task['raw_json_path']):
            print(f"  Skipping: Result file already exists -> '{os.path.basename(task['raw_json_path'])}'")
            continue

        print(f" Getting results for: Location='{task['location_name']}', Keyword='{task['keyword']}'")
        get_dataforseo_results(task['task_id'], task['api_type'], task['raw_json_path'])
        time.sleep(1)

    print("\nAll result retrieval attempts completed.")

# Process Data

In [ ]:
KEYWORD_CATEGORIES = {
    'keywords_General_Dentist': [
        'dental clinic', 'family dentist', 'dentist',
        'general dentistry', 'cosmetic dentist',
    ],
    'keywords_Special_Dentist': [
        'orthodontist', 'pediatric dentist', 'periodontist', 'prosthodontist',
    ],
    'keywords_Surgery_Dentist': [
        'oral surgeon', 'dental implants', 'emergency dentist',
    ]
}

keyword_to_category_map = {
    keyword: category
    for category, keywords in KEYWORD_CATEGORIES.items()
    for keyword in keywords
}


def get_category_from_keywords(keyword_list):
    for keyword in keyword_list:
        if keyword in keyword_to_category_map:
            return keyword_to_category_map[keyword]
    return 'Unknown'

In [ ]:
LOCATION_PREFIXES = ['Malone_NY_S_', 'SaranacLake_NY_S_', 'Syracuse_NY_M_','Buffalo_NY_L_','NYC_NY_L_',
                      'Eureka_CA_S_', 'FortBragg_CA_S_', 'Modesto_CA_M_', 'SanFrancisco_CA_L_','LA_CA_L_',
                      'Vidalia_GA_S_', 'Toccoa_GA_S_', 'Macon_GA_M_', 'Atlanta_GA_L_', 'Augusta_GA_L_']
def parse_local_finder_results(file_path):
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
    except (FileNotFoundError, json.JSONDecodeError) as e:
        print(f"Could not read or parse {os.path.basename(file_path)}: {e}")
        return []

    file_name = os.path.basename(file_path)
    search_location = "Unknown"
    try:
        keywords_str = file_name

        # remove LOCATION_PREFIXES
        for prefix in LOCATION_PREFIXES:
            if keywords_str.startswith(prefix):
                search_location = prefix.rstrip('_')
                keywords_str = keywords_str.replace(prefix, '', 1)
                break

        # Clean the suffixes
        keywords_str = keywords_str.replace('_local_finder.json', '').replace('_maps.json', '')

        # Extract keywords
        keywords_list = [kw.replace('_', ' ') for kw in keywords_str.split('+')]
        formatted_keywords = ', '.join(keywords_list)
        category = get_category_from_keywords(keywords_list)

    except Exception as e:
        print(f"Error processing filename {file_name}: {e}")
        formatted_keywords = 'N/A'
        category = 'Unknown'

    extracted_data = []
    if not (data and data.get("tasks") and data["tasks"][0].get("result")):
        return []

    for result in data["tasks"][0]["result"]:
        if not result or not result.get("items"):
            continue
        for item in result["items"]:
            rating = item.get("rating", {})
            if not isinstance(rating, dict): rating = {}

            extracted_data.append({
                "title": item.get("title"),
                "description": item.get("description"),
                "rating_value": rating.get("value"),
                "votes_count": rating.get("votes_count"),
                "type": item.get("type"),
                "keywords": formatted_keywords,
                "category": category,
                "search_location": search_location
            })
    return extracted_data

def parse_maps_results(file_path):
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
    except (FileNotFoundError, json.JSONDecodeError) as e:
        print(f"Could not read or parse {os.path.basename(file_path)}: {e}")
        return []


    file_name = os.path.basename(file_path)
    search_location = "Unknown"
    try:
        keywords_str = file_name

        for prefix in LOCATION_PREFIXES:
            if keywords_str.startswith(prefix):
                search_location = prefix.rstrip('_')
                keywords_str = keywords_str.replace(prefix, '', 1)
                break

        keywords_str = keywords_str.replace('_local_finder.json', '').replace('_maps.json', '')

        keywords_list = [kw.replace('_', ' ') for kw in keywords_str.split('+')]
        formatted_keywords = ', '.join(keywords_list)
        category = get_category_from_keywords(keywords_list)

    except Exception as e:
        print(f"Error processing filename {file_name}: {e}")
        formatted_keywords = 'N/A'
        category = 'Unknown'

    extracted_data = []
    if not (data and data.get("tasks") and data["tasks"][0].get("result")):
        return []

    for result in data["tasks"][0]["result"]:
        if not result or not result.get("items"):
            continue
        for item in result["items"]:
            rating = item.get("rating", {})
            if not isinstance(rating, dict): rating = {}
            rating_distribution = item.get("rating_distribution", {})
            if not isinstance(rating_distribution, dict): rating_distribution = {}
            address_info = item.get("address_info", {})
            if not isinstance(address_info, dict): address_info = {}

            extracted_data.append({
                "title": item.get("title"),
                "address": item.get("address"),
                "latitude": item.get("latitude"),
                "longitude": item.get("longitude"),
                "zip": address_info.get("zip"),
                "rating_value": rating.get("value"),
                "votes_count": rating.get("votes_count"),
                "rating_1_star": rating_distribution.get("1", 0),
                "rating_2_star": rating_distribution.get("2", 0),
                "rating_3_star": rating_distribution.get("3", 0),
                "rating_4_star": rating_distribution.get("4", 0),
                "rating_5_star": rating_distribution.get("5", 0),
                "type": item.get("type"),
                "keywords": formatted_keywords,
                "category": category,
                "search_location": search_location
            })
    return extracted_data


In [ ]:
print("--- Data Processing  ---")

temp_dir = os.path.join(base_path, "temp")
output_dir = os.path.join(base_path, "output")

local_finder_raw_json_dir = os.path.join(temp_dir, "local_finder")
maps_raw_json_dir = os.path.join(temp_dir, "maps")

local_finder_output_raw_dir = os.path.join(output_dir, "local_finder", "raw")
local_finder_output_processed_dir = os.path.join(output_dir, "local_finder", "processed")
maps_output_raw_dir = os.path.join(output_dir, "maps", "raw")
maps_output_processed_dir = os.path.join(output_dir, "maps", "processed")

for d in [local_finder_output_raw_dir, local_finder_output_processed_dir, maps_output_raw_dir, maps_output_processed_dir]:
    os.makedirs(d, exist_ok=True)

# log file
local_finder_log_path = os.path.join(local_finder_raw_json_dir, "_processed_files.log")
maps_log_path = os.path.join(maps_raw_json_dir, "_processed_files.log")

def load_processed_log(log_path):
    # Load files already processed
    try:
        with open(log_path, 'r') as f:
            return set(line.strip() for line in f)
    except FileNotFoundError:
        return set()

def update_processed_log(log_path, new_files):
    # Append newly processed filenames to the log
    with open(log_path, 'a') as f:
        for file_name in new_files:
            f.write(f"{file_name}\n")

def load_existing_data(csv_path):
    # Load the previously processed CSV data
    try:
        return pd.read_csv(csv_path)
    except (FileNotFoundError, pd.errors.EmptyDataError):
        return pd.DataFrame()

# Load Logs
processed_lf_set = load_processed_log(local_finder_log_path)
processed_maps_set = load_processed_log(maps_log_path)
print(f"Loaded {len(processed_lf_set)} processed Local Finder file records from log.")
print(f"Loaded {len(processed_maps_set)} processed Maps file records from log.")

local_finder_json_files = glob.glob(os.path.join(local_finder_raw_json_dir, "*.json"))
maps_json_files = glob.glob(os.path.join(maps_raw_json_dir, "*.json"))

if not local_finder_json_files and not maps_json_files:
    print("No JSON result files found for processing in temp/local_finder or temp/maps.")
else:
    print(f"Found {len(local_finder_json_files)} total Local Finder files and {len(maps_json_files)} total Maps files.")

    # local finder
    new_local_finder_data = []
    processed_this_run_lf = []

    for file_path in local_finder_json_files:
        file_name = os.path.basename(file_path)
        if file_name in processed_lf_set:
            continue  # Skip if in log

        print(f"Processing new Local Finder file: {file_name}")
        parsed_data = parse_local_finder_results(file_path)
        if parsed_data:
            new_local_finder_data.extend(parsed_data)
            processed_this_run_lf.append(file_name)

    # map
    new_maps_data = []
    processed_this_run_maps = []

    for file_path in maps_json_files:
        file_name = os.path.basename(file_path)
        if file_name in processed_maps_set:
            continue  # Skip if in log

        print(f"Processing new Maps file: {file_name}")
        parsed_data = parse_maps_results(file_path)
        if parsed_data:
            new_maps_data.extend(parsed_data)
            processed_this_run_maps.append(file_name)

    # Consolidate Local Finder data
    local_finder_processed_csv_path = os.path.join(local_finder_output_processed_dir, "local_finder_processed_summary.csv")

    if new_local_finder_data:
        df_new_lf = pd.DataFrame(new_local_finder_data)
        df_old_lf = load_existing_data(local_finder_processed_csv_path)
        df_all_lf = pd.concat([df_old_lf, df_new_lf], ignore_index=True)
        print(f"\nLoaded {len(df_old_lf)} old Local Finder records. {len(df_new_lf)} new records added.")

        local_finder_raw_csv_path = os.path.join(local_finder_output_raw_dir, "local_finder_raw_summary.csv")
        df_all_lf.to_csv(local_finder_raw_csv_path, index=False, encoding='utf-8-sig')
        print(f"Local Finder locations before deduplication: {len(df_all_lf)}")

        subset_cols = ['title']
        if 'search_location' in df_all_lf.columns:
            subset_cols.append('search_location')

        local_finder_deduplicated_df = df_all_lf.drop_duplicates(subset=subset_cols, keep='first')
        print(f"Local Finder locations after deduplication: {len(local_finder_deduplicated_df)}")
        local_finder_sorted_df = local_finder_deduplicated_df.sort_values(by=['rating_value', 'votes_count'], ascending=[False, False], na_position='last')

        local_finder_sorted_df.to_csv(local_finder_processed_csv_path, index=False, encoding='utf-8-sig')
        print(f"\nSaved deduplicated Local Finder data to: '{local_finder_processed_csv_path}'")
        display(local_finder_sorted_df.head(10))

        update_processed_log(local_finder_log_path, processed_this_run_lf)
        print(f"Updated Local Finder log with {len(processed_this_run_lf)} new files.")
    else:
        print("\nNo new Local Finder files to process.")

    # Consolidate Maps data
    maps_processed_csv_path = os.path.join(maps_output_processed_dir, "maps_processed_summary.csv")

    if new_maps_data:
        df_new_maps = pd.DataFrame(new_maps_data)
        df_old_maps = load_existing_data(maps_processed_csv_path)
        df_all_maps = pd.concat([df_old_maps, df_new_maps], ignore_index=True)
        print(f"\nLoaded {len(df_old_maps)} old Maps records. {len(df_new_maps)} new records added.")

        maps_raw_csv_path = os.path.join(maps_output_raw_dir, "maps_raw_summary.csv")
        df_all_maps.to_csv(maps_raw_csv_path, index=False, encoding='utf-8-sig')
        print(f"Maps locations before deduplication: {len(df_all_maps)}")

        subset_cols_maps = ['title', 'address']
        if 'search_location' in df_all_maps.columns:
            subset_cols_maps.append('search_location')

        maps_deduplicated_df = df_all_maps.drop_duplicates(subset=subset_cols_maps, keep='first')

        print(f"Maps locations after deduplication: {len(maps_deduplicated_df)}")
        maps_sorted_df = maps_deduplicated_df.sort_values(by=['rating_value', 'votes_count'], ascending=[False, False], na_position='last')

        maps_sorted_df.to_csv(maps_processed_csv_path, index=False, encoding='utf-8-sig')
        print(f"\nSuccessfully saved deduplicated Maps data to: '{maps_processed_csv_path}'")
        display(maps_sorted_df.head(10))

        update_processed_log(maps_log_path, processed_this_run_maps)
        print(f"Updated Maps log with {len(processed_this_run_maps)} new files.")
    else:
        print("\nNo new Maps files to process.")

    print("\nAll result processing attempts completed.")

# Merge and Deduplicate Processed Results

In [ ]:
print("--- Merging Processed Results ---")

output_dir = os.path.join(base_path, "output")
local_finder_processed_path = os.path.join(output_dir, "local_finder", "processed", "local_finder_processed_summary.csv")
maps_processed_path = os.path.join(output_dir, "maps", "processed", "maps_processed_summary.csv")

local_finder_processed_df = pd.DataFrame()
maps_processed_df = pd.DataFrame()

try:
    local_finder_processed_df = pd.read_csv(local_finder_processed_path)
    print(f"Loaded Local Finder processed data from: '{local_finder_processed_path}'")
except FileNotFoundError:
    print(f"Error: '{local_finder_processed_path}' not found.")

try:
    maps_processed_df = pd.read_csv(maps_processed_path)
    print(f"Loaded Maps processed data from: '{maps_processed_path}'")
except FileNotFoundError:
    print(f"Error: '{maps_processed_path}' not found.")

# Check if both df were loaded successfully
if not local_finder_processed_df.empty and not maps_processed_df.empty:
    if 'search_location' in local_finder_processed_df.columns:
        local_finder_processed_df['search_location'] = local_finder_processed_df['search_location'].astype(str)
    if 'search_location' in maps_processed_df.columns:
        maps_processed_df['search_location'] = maps_processed_df['search_location'].astype(str)

    merge_keys = ['title']
    if 'search_location' in local_finder_processed_df.columns and 'search_location' in maps_processed_df.columns:
        merge_keys.append('search_location')
        print(f"Merging on keys: {merge_keys}")
    else:
        print("Warning: 'search_location' missing in one of the dataframes. Merging on 'title' only (Risk of cross-city mismatch).")

    merged_df = pd.merge(
        local_finder_processed_df,
        maps_processed_df,
        on=merge_keys,
        how='outer',
        suffixes=('_local_finder', '_maps')
    )
    print(f"Merged df shape: {merged_df.shape}")

    # Create indicator columns based is in which or both api
    merged_df['isFinder'] = merged_df['rating_value_local_finder'].apply(lambda x: 0 if pd.isna(x) else 1)
    merged_df['isMap'] = merged_df['rating_value_maps'].apply(lambda x: 0 if pd.isna(x) else 1)
    merged_df['isBoth'] = merged_df.apply(lambda row: 1 if row['isFinder'] == 1 and row['isMap'] == 1 else 0, axis=1)

    merged_df['keywords'] = merged_df['keywords_local_finder'].fillna(merged_df['keywords_maps'])
    merged_df['category'] = merged_df['category_local_finder'].fillna(merged_df['category_maps'])

    merged_df['votes_count'] = merged_df['votes_count_local_finder'].fillna(merged_df['votes_count_maps'])
    merged_df['rating_value'] = merged_df['rating_value_local_finder'].fillna(merged_df['rating_value_maps'])

    for i in range(1, 6):
        star_col = f'rating_{i}_star'
        if star_col in merged_df.columns:
            merged_df[star_col] = merged_df[star_col].fillna(0)
        else:
            merged_df[star_col] = 0

    final_df = merged_df[[
        'title', 'keywords', 'category','search_location',
        'votes_count', 'rating_value', 'address', 'latitude', 'longitude','zip',
        'rating_1_star', 'rating_2_star', 'rating_3_star', 'rating_4_star', 'rating_5_star',
        'isFinder', 'isMap', 'isBoth'
    ]].copy()


    print(f"\nCombined df shape before deduplication: {final_df.shape}")

    # Deduplicate the combined df based on 'title'
    dedup_subset = ['title']
    if 'search_location' in final_df.columns:
        dedup_subset.append('search_location')

    final_deduplicated_df = final_df.drop_duplicates(subset=dedup_subset, keep='first')

    print(f"Combined df shape after deduplication: {final_deduplicated_df.shape}")

    # Sort the final df by rating_value and votes_count
    final_sorted_df = final_deduplicated_df.sort_values(by=['rating_value', 'votes_count'], ascending=[False, False], na_position='last')

    print("\nFinal Data Preview:")
    display(final_sorted_df.head(10))


    # save
    final_output_dir = os.path.join(output_dir, "merged_deduplicated")
    if not os.path.exists(final_output_dir):
        os.makedirs(final_output_dir)
        print(f"Created directory: {final_output_dir}")

    final_csv_path = os.path.join(final_output_dir, "merged_deduplicated_summary.csv")
    final_sorted_df.to_csv(final_csv_path, index=False, encoding='utf-8-sig') # Save sorted dataframe
    print(f"\nSaved final data to: '{final_csv_path}'")


elif local_finder_processed_df.empty:
    print("\nCannot merge. Local Finder processed data not found.")
elif maps_processed_df.empty:
     print("\nCannot merge. Maps processed data not found.")

# Get isFinder==1 but isMap==0 as new keyword and POST task

In [ ]:
print("--- Posting Supplementary Tasks ---")

output_dir = os.path.join(base_path, "output")
input_dir = os.path.join(base_path, "input")
temp_dir = os.path.join(base_path, "temp")
merged_deduplicated_path = os.path.join(output_dir, "merged_deduplicated", "merged_deduplicated_summary.csv")
supplement_task_list_csv_path = os.path.join(temp_dir, "supplement_task_list.csv")

mapping_data = {
    'location_name': [
        'Malone_NY_S', 'SaranacLake_NY_S', 'Syracuse_NY_M','Buffalo_NY_L','NYC_NY_L',
        'Eureka_CA_S', 'FortBragg_CA_S', 'Modesto_CA_M', 'SanFrancisco_CA_L','LA_CA_L',
        'Vidalia_GA_S', 'Toccoa_GA_S', 'Macon_GA_M', 'Atlanta_GA_L', 'Augusta_GA_L'
    ],
    'location_code': [
        1026588, 1023342, 1027001, 1022764, 1023191,
        1013774, 1013806, 1014019, 1014221, 1013962,
        1015545, 1015533, 1015427, 1015254, 1015256
    ]
}

location_map = dict(zip(mapping_data['location_name'], mapping_data['location_code']))
print(f"Built location map for {len(location_map)} cities using provided sample data.")

existing_tasks = set()
try:
    existing_tasks_df = pd.read_csv(supplement_task_list_csv_path)
    for index, row in existing_tasks_df.iterrows():
        existing_tasks.add((row['location_name'], row['keyword']))
    print(f"Loaded {len(existing_tasks)} existing supplementary tasks to skip.")
except FileNotFoundError:
    print("No existing supplementary task file found. Will post all tasks.")



try:
    merged_df = pd.read_csv(merged_deduplicated_path)
    if 'search_location' in merged_df.columns:
        merged_df['search_location'] = merged_df['search_location'].astype(str)
    else:
        print("CRITICAL ERROR: 'search_location' column missing in merged CSV.")
        merged_df = pd.DataFrame()

    print(f"Loaded merged data from: '{merged_deduplicated_path}'")
except FileNotFoundError:
    print(f"Error: '{merged_deduplicated_path}' not found.")
    merged_df = pd.DataFrame()

if not merged_df.empty and location_map:
    missing_df = merged_df[(merged_df['isFinder'] == 1) & (merged_df['isMap'] == 0)].copy()

    missing_df = missing_df.dropna(subset=['title', 'search_location'])

    total_missing = len(missing_df)
    print(f"Found {total_missing} clinics present in Finder but missing in Maps details.")

    if total_missing > 0:
        maps_api_url = "https://api.dataforseo.com/v3/serp/google/maps/task_post"
        supplement_maps_output_dir = os.path.join(temp_dir, "maps_supplement")
        os.makedirs(supplement_maps_output_dir, exist_ok=True)


        write_header = not os.path.exists(supplement_task_list_csv_path)

        grouped = missing_df.groupby('search_location')

        for city_name, group_data in grouped:
            location_code = location_map.get(city_name)

            if not location_code:
                print(f"Skipping city '{city_name}': Code not found in provided mapping.")
                continue

            clinics_to_search = group_data['title'].unique().tolist()

            print(f"\nProcessing city '{city_name}' (Code: {location_code}) - {len(clinics_to_search)} missing clinics:")

            for clinic_keyword in clinics_to_search:
                if (city_name, clinic_keyword) in existing_tasks:
                    # print(f"  Skipping existing: {clinic_keyword}")
                    continue


                task_id = post_dataforseo_task(maps_api_url, location_code, clinic_keyword)

                if task_id:
                    safe_keyword = str(clinic_keyword).replace(' ', '_').replace('/', '_').replace('\\', '_')
                    raw_json_filename = f"{city_name}_{safe_keyword}_maps.json"
                    raw_json_path = os.path.join(supplement_maps_output_dir, raw_json_filename)

                    current_task_df = pd.DataFrame([{
                        "task_id": task_id,
                        "api_type": "maps",
                        "location_name": city_name,
                        "search_location": city_name,
                        "keyword": clinic_keyword,
                        "raw_json_path": raw_json_path
                    }])

                    current_task_df.to_csv(supplement_task_list_csv_path, mode='a', header=write_header, index=False)
                    write_header = False #

                    print(f"  Posted: '{clinic_keyword}'")

                time.sleep(1)

        print(f"\nAll supplementary tasks posted.")
        print("!!! IMPORTANT: Please wait 20 minutes before running the result retrieval script. !!!")

    else:
        print("No missing Maps data found (all Local Finder items have Maps details).")
else:
    print("Cannot proceed: Merged dataframe is empty or Location Map is empty.")


# Get supplement result

In [ ]:
print("--- Get & Process Supplementary Task Results ---")

temp_dir = os.path.join(base_path, "temp")
output_dir = os.path.join(base_path, "output")
supplement_task_list_path = os.path.join(temp_dir, "supplement_task_list.csv")

# Get Task Results
tasks_to_get = []
try:
    supplement_task_df = pd.read_csv(supplement_task_list_path)
    tasks_to_get = supplement_task_df.to_dict('records')
    print(f"Found {len(tasks_to_get)} supplementary tasks to get results for.")

    for task in tasks_to_get:
        # Skip existing files
        if os.path.exists(task['raw_json_path']):
            print(f"  Skipping: Result file already exists -> '{os.path.basename(task['raw_json_path'])}'")
            continue

        print(f" Getting results for: Location='{task['location_name']}', Keyword='{task['keyword']}'")
        get_dataforseo_results(task['task_id'], task['api_type'], task['raw_json_path'])
        time.sleep(0.5)

    print("\nAll supplementary result retrieval attempts completed.")

except FileNotFoundError:
    print(f"Error: '{supplement_task_list_path}' not found. No supplementary tasks to process.")




In [ ]:
# Process Data
if tasks_to_get:
    maps_supplement_raw_json_dir = os.path.join(temp_dir, "maps_supplement")

    # log file to skip existing
    processed_log_path = os.path.join(maps_supplement_raw_json_dir, "processed_files.log")
    try:
        with open(processed_log_path, 'r') as f:
            processed_files_set = set(line.strip() for line in f)
        print(f"Loaded {len(processed_files_set)} records from the processed file log.")
    except FileNotFoundError:
        processed_files_set = set()
        print("No processed file log found. Will process all files.")

    supplement_json_files = glob.glob(os.path.join(maps_supplement_raw_json_dir, "*.json"))
    new_supplement_data = []
    files_processed_this_run = []

    print(f"\nFound {len(supplement_json_files)} total supplementary JSON files.")

    for file_path in supplement_json_files:
        file_name = os.path.basename(file_path)

        # skip if in log
        if file_name in processed_files_set:
            print(f"  Skipping already processed file: {file_name}")
            continue

        print(f"  Processing new supplementary file: {file_name}")

        # process resuly
        parsed_data = parse_maps_results(file_path)
        if parsed_data:
            new_supplement_data.extend(parsed_data)
            files_processed_this_run.append(file_name)
        else:
            print(f"  No data extracted from {file_name}.")

    # merge , deduplicate
    if new_supplement_data:
        df_new = pd.DataFrame(new_supplement_data)
        print(f"\nProcessed {len(df_new)} new records from {len(files_processed_this_run)} new files.")
        supplement_output_dir = os.path.join(output_dir, "supplement")
        os.makedirs(supplement_output_dir, exist_ok=True)
        supplement_csv_path = os.path.join(supplement_output_dir, "supplement_processed_summary.csv")

        try:
            df_old = pd.read_csv(supplement_csv_path)
            print(f"Loaded {len(df_old)} existing records from CSV.")
            if 'search_location' in df_new.columns and 'search_location' not in df_old.columns:
                df_old['search_location'] = 'Unknown'

            df_all = pd.concat([df_old, df_new], ignore_index=True)
        except (FileNotFoundError, pd.errors.EmptyDataError):
            print("No existing CSV found. Using new data only.")
            df_all = df_new

        # deduplicate, sort
        print(f"Total records before deduplication: {len(df_all)}")

        dedup_subset = ['title', 'address']
        if 'search_location' in df_all.columns:
            dedup_subset.append('search_location')

        supplement_deduplicated_df = df_all.drop_duplicates(subset=dedup_subset, keep='first')

        supplement_sorted_df = supplement_deduplicated_df.sort_values(by=['rating_value', 'votes_count'], ascending=[False, False], na_position='last')
        print(f"After deduplication, {len(supplement_sorted_df)} unique records remain.")

        # Save
        supplement_sorted_df.to_csv(supplement_csv_path, index=False, encoding='utf-8-sig')
        print(f"\nSaved final processed supplementary data to: '{supplement_csv_path}'")
        display(supplement_sorted_df.head())

        # Update log
        with open(processed_log_path, 'a') as f:
            for file_name in files_processed_this_run:
                f.write(f"{file_name}\n")
        print(f"Updated 'processed_files.log' with {len(files_processed_this_run)} new filenames.")

    else:
        print("\nNo new supplementary files to process.")
else:
    print("\nNo supplementary tasks were run (task list empty and no json files found), skipping processing.")

# Merge and process

In [ ]:
print("--- Final Merging and Consolidation ---")

output_dir = os.path.join(base_path, "output")

original_merged_path = os.path.join(output_dir, "merged_deduplicated", "merged_deduplicated_summary.csv")
supplement_path = os.path.join(output_dir, "supplement", "supplement_processed_summary.csv")

try:
    original_df = pd.read_csv(original_merged_path)
    print(f"Loaded original merged data: {original_df.shape}")
except FileNotFoundError:
    print(f"Error: '{original_merged_path}' not found.")
    original_df = pd.DataFrame()

try:
    supplement_df = pd.read_csv(supplement_path)
    print(f"Loaded supplementary data: {supplement_df.shape}")
except FileNotFoundError:
    print(f"'{supplement_path}' not found.")
    supplement_df = pd.DataFrame()

def ensure_location_col(df):
    if df.empty: return df
    if 'search_location' not in df.columns:
        print("Wait, 'search_location' is missing in source df. Filling with 'Unknown'.")
        df['search_location'] = 'Unknown'
    df['search_location'] = df['search_location'].fillna('Unknown').astype(str)
    return df

if not original_df.empty:

    original_df = ensure_location_col(original_df)

    if not supplement_df.empty:
        print("\nMerging with supplementary data...")


        supplement_df = ensure_location_col(supplement_df)


        supplement_df['isFinder'] = 0
        supplement_df['isMap'] = 1
        supplement_df['isBoth'] = 0


        combined_df = pd.concat([original_df, supplement_df], ignore_index=True, sort=False)
        combined_df = ensure_location_col(combined_df)

        print(f"Shape before grouping: {combined_df.shape}")


        group_keys = ['title', 'search_location']

        agg_cols = [c for c in combined_df.columns if c not in group_keys]
        agg_funcs = {col: 'first' for col in agg_cols}

        agg_funcs['isFinder'] = 'max'
        agg_funcs['isMap'] = 'max'

        final_df = combined_df.groupby(group_keys).agg(agg_funcs)

        final_df['isBoth'] = ((final_df['isFinder'] == 1) & (final_df['isMap'] == 1)).astype(int)

        final_df = final_df.reset_index()

        print(f"Shape after consolidation: {final_df.shape}")

    else:
        print("\nNo supplementary data to merge. Using original data.")
        final_df = original_df

    if 'search_location' not in final_df.columns:
        print("CRITICAL ERROR: Column still missing. Creating manually from Unknown.")
        final_df['search_location'] = 'Unknown'

    sort_cols = ['isBoth', 'rating_value', 'votes_count']
    valid_sort_cols = [c for c in sort_cols if c in final_df.columns]

    final_sorted_df = final_df.sort_values(
        by=valid_sort_cols,
        ascending=[False] * len(valid_sort_cols),
        na_position='last'
    )

    final_output_dir = os.path.join(output_dir, "final")
    os.makedirs(final_output_dir, exist_ok=True)

    final_csv_path = os.path.join(final_output_dir, "final_processed.csv")
    final_sorted_df.to_csv(final_csv_path, index=False, encoding='utf-8-sig')

    print(f"\nSaved final processed data to: '{final_csv_path}'")

    print("\nFinal Processed Data Preview:")
    display(final_sorted_df.head(10))

else:
    print("Original merged data not found.")

# Rate Distribution

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np

final_csv_path = "/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/output/final/final_processed.csv"


try:
    df = pd.read_csv(final_csv_path)
    print("Successfully loaded final_processed.csv")
except FileNotFoundError:
    print(f"Error: The file '{final_csv_path}' was not found. Please check the path.")
    df = pd.DataFrame()

if not df.empty:
    if 'search_location' not in df.columns:
        if 'location_name' in df.columns:
            print("Warning: 'search_location' column missing. Using 'location_name' instead.")
            df['search_location'] = df['location_name']
        else:
            print("Error: Neither 'search_location' nor 'location_name' found.")
            df = pd.DataFrame()

if not df.empty:
    df_filtered = df[(df['category'] != 'Unknown') & (df['search_location'] != 'Unknown')].copy()

    df_filtered['rating_value'] = pd.to_numeric(df_filtered['rating_value'], errors='coerce')
    df_filtered.dropna(subset=['rating_value', 'category', 'search_location'], inplace=True)

    # order (SSMLL)
    state_orders = {
        'NY': ['Malone_NY_S', 'SaranacLake_NY_S', 'Syracuse_NY_M', 'Buffalo_NY_L', 'NYC_NY_L'],
        'CA': ['Eureka_CA_S', 'FortBragg_CA_S', 'Modesto_CA_M', 'SanFrancisco_CA_L', 'LA_CA_L'],
        'GA': ['Vidalia_GA_S', 'Toccoa_GA_S', 'Macon_GA_M', 'Augusta_GA_L', 'Atlanta_GA_L']
    }

    def get_state(loc):
        loc_str = str(loc)
        if '_NY_' in loc_str: return 'NY'
        if '_CA_' in loc_str: return 'CA'
        if '_GA_' in loc_str: return 'GA'
        return 'Other'

    df_filtered['State'] = df_filtered['search_location'].apply(get_state)

    target_states = ['NY', 'CA', 'GA']
    df_filtered = df_filtered[df_filtered['State'].isin(target_states)]

    print(f"Records remaining after filtering for NY, CA, GA: {len(df_filtered)}")

    # Low Rating Dummy
    df_filtered['low_rating_dummy'] = np.where(df_filtered['rating_value'] <= 3, 1, 0)

    # type
    def get_location_type(loc):
        if str(loc).endswith('_S'): return 'Country/Small Town'
        return 'Urban/Metro'

    df_filtered['location_type'] = df_filtered['search_location'].apply(get_location_type)

    # std
    def rating_std(row):
        all_ratings = []
        try:
            all_ratings.extend([1] * int(row.get('rating_1_star', 0)))
            all_ratings.extend([2] * int(row.get('rating_2_star', 0)))
            all_ratings.extend([3] * int(row.get('rating_3_star', 0)))
            all_ratings.extend([4] * int(row.get('rating_4_star', 0)))
            all_ratings.extend([5] * int(row.get('rating_5_star', 0)))

            if len(all_ratings) < 2:
                return 0
            return pd.Series(all_ratings).std()
        except Exception:
            return np.nan

    star_cols = ['rating_1_star', 'rating_2_star', 'rating_3_star', 'rating_4_star', 'rating_5_star']
    for col in star_cols:
        if col in df_filtered.columns:
            df_filtered[col] = pd.to_numeric(df_filtered[col], errors='coerce').fillna(0)
        else:
            df_filtered[col] = 0

    df_filtered['rating_std'] = df_filtered.apply(rating_std, axis=1)


    categories = sorted(df_filtered['category'].unique())

    # A. Boxplots
    for category in categories:
        print(f"Plotting Boxplots for Category: {category}")
        for state in target_states:
            plot_data = df_filtered[(df_filtered['category'] == category) & (df_filtered['State'] == state)]

            if plot_data.empty:
                continue

            plt.figure(figsize=(10, 6))
            sns.boxplot(
                x='search_location',
                y='rating_value',
                data=plot_data,
                order=state_orders[state],
                palette='Set2',
                hue='search_location',
                legend=False
            )

            plt.title(f'{state} - Rating Distribution: {category}', fontsize=16)
            plt.xlabel('Location', fontsize=12)
            plt.ylabel('Rating Value', fontsize=12)
            plt.ylim(1, 5.2)
            plt.xticks(rotation=15)
            plt.grid(axis='y', linestyle='--', alpha=0.7)
            plt.tight_layout()
            plt.show()

    # B. Histograms
    for category in categories:
        for state in target_states:
            locations = state_orders[state]
            for location in locations:
                plot_df = df_filtered[
                    (df_filtered['category'] == category) &
                    (df_filtered['search_location'] == location)
                ]

                if plot_df.empty:
                    continue

                plt.figure(figsize=(8, 5))
                sns.histplot(data=plot_df, x='rating_value', bins=10, kde=True, color='blue')

                plt.title(f'{category} in {location} ({state})')
                plt.xlabel('Rating Value')
                plt.ylabel('Count')
                plt.xlim(1, 5)
                plt.tight_layout()
                plt.show()

    # C. Violin Plots
    for category in categories:
        for state in target_states:
            plot_data = df_filtered[(df_filtered['category'] == category) & (df_filtered['State'] == state)]
            if plot_data.empty: continue

            plt.figure(figsize=(10, 6))
            sns.violinplot(
                x='search_location',
                y='rating_value',
                data=plot_data,
                order=state_orders[state],
                palette='Pastel1',
                hue='search_location',
                legend=False
            )

            plt.title(f'{state} - Violin Plot: {category}', fontsize=16)
            plt.xlabel('Location', fontsize=12)
            plt.ylabel('Rating Value', fontsize=12)
            plt.xticks(rotation=15)
            plt.tight_layout()
            plt.show()

    # D. Low Rating Analysis
    print("\n Low Rating Analysis (Urban vs Rural)")
    low_rating_crosstab = pd.crosstab(df_filtered['location_type'], df_filtered['low_rating_dummy'])
    print(low_rating_crosstab)

    low_rating_crosstab.plot(kind='bar', stacked=True, figsize=(8, 6), color=['skyblue', 'salmon'])
    plt.title('Count of Low Ratings (<=3) by Location Type')
    plt.xlabel('Location Type')
    plt.ylabel('Count of Clinics')
    plt.xticks(rotation=0)
    plt.legend(title='Rating <= 3', labels=['No', 'Yes'])
    plt.tight_layout()
    plt.show()

    # Std Bar Plots
    std_summary = df_filtered.groupby(['category', 'search_location', 'State'], as_index=False)['rating_std'].mean()

    for category in categories:
        for state in target_states:
            plot_data = std_summary[(std_summary['category'] == category) & (std_summary['State'] == state)]
            if plot_data.empty: continue

            plt.figure(figsize=(10, 6))
            sns.barplot(
                x='search_location',
                y='rating_std',
                data=plot_data,
                order=state_orders[state],
                palette='coolwarm',
                hue='search_location',
                legend=False
            )

            plt.title(f'{state} - Avg Rating Std Dev: {category}', fontsize=16)
            plt.xlabel('Location', fontsize=12)
            plt.ylabel('Avg Std Dev of Ratings', fontsize=12)
            plt.xticks(rotation=15)
            plt.tight_layout()
            plt.show()

else:
    print("DataFrame is empty. Cannot generate plots.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np
import math

final_csv_path = "/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/output/final/final_processed.csv"


try:
    df = pd.read_csv(final_csv_path)
    print("Successfully loaded final_processed.csv")
except FileNotFoundError:
    print(f"Error: The file '{final_csv_path}' was not found.")
    df = pd.DataFrame()

def get_fig_title(title):
    return title

if not df.empty:
    if 'search_location' not in df.columns:
        if 'location_name' in df.columns:
            print("Warning: 'search_location' column missing. Using 'location_name' instead.")
            df['search_location'] = df['location_name']
        else:
            print("Error: Neither 'search_location' nor 'location_name' found.")
            df = pd.DataFrame()

if not df.empty:
    state_orders = {
        'NY': ['Malone_NY_S', 'SaranacLake_NY_S', 'Syracuse_NY_M', 'Buffalo_NY_L', 'NYC_NY_L'],
        'CA': ['Eureka_CA_S', 'FortBragg_CA_S', 'Modesto_CA_M', 'SanFrancisco_CA_L', 'LA_CA_L'],
        'GA': ['Vidalia_GA_S', 'Toccoa_GA_S', 'Macon_GA_M', 'Augusta_GA_L', 'Atlanta_GA_L']
    }


    all_target_cities = [city for cities in state_orders.values() for city in cities]


    df['search_location'] = df['search_location'].astype(str)


    df_filtered = df[
        (df['category'] != 'Unknown') &
        (df['search_location'].isin(all_target_cities))
    ].copy()


    df_filtered['location_name'] = df_filtered['search_location']


    def get_state(loc):
        if '_NY_' in loc: return 'NY'
        if '_CA_' in loc: return 'CA'
        if '_GA_' in loc: return 'GA'
        return 'Other'

    df_filtered['State'] = df_filtered['location_name'].apply(get_state)

    print(f"Records remaining after filtering target cities: {len(df_filtered)}")


    df_filtered['rating_value'] = pd.to_numeric(df_filtered['rating_value'], errors='coerce')
    df_filtered.dropna(subset=['rating_value', 'category', 'location_name'], inplace=True)


    star_cols = ['rating_1_star', 'rating_2_star', 'rating_3_star', 'rating_4_star', 'rating_5_star']
    for col in star_cols:
        if col in df_filtered.columns:
            df_filtered[col] = pd.to_numeric(df_filtered[col], errors='coerce').fillna(0).astype(int)
        else:
            df_filtered[col] = 0


    def get_location_type(loc):
        if str(loc).endswith('_S'): return 'Small Town/Rural'
        if str(loc).endswith('_M'): return 'Mid-Size City'
        return 'Urban/Metro'

    df_filtered['location_type'] = df_filtered['location_name'].apply(get_location_type)

    def calculate_metrics(row):
        ratings_distribution = []
        try:
            ratings_distribution.extend([1] * row['rating_1_star'])
            ratings_distribution.extend([2] * row['rating_2_star'])
            ratings_distribution.extend([3] * row['rating_3_star'])
            ratings_distribution.extend([4] * row['rating_4_star'])
            ratings_distribution.extend([5] * row['rating_5_star'])
        except:
             return pd.Series([np.nan, np.nan, np.nan, np.nan])

        if not ratings_distribution:
            return pd.Series([np.nan, np.nan, np.nan, np.nan])

        std_dev = np.std(ratings_distribution, ddof=1) if len(ratings_distribution) > 1 else 0
        p10 = np.percentile(ratings_distribution, 10)
        p25 = np.percentile(ratings_distribution, 25)
        p75 = np.percentile(ratings_distribution, 75)
        iqr = p75 - p25

        return pd.Series([std_dev, p10, p25, iqr])

    metrics_df = df_filtered.apply(calculate_metrics, axis=1)
    metrics_df.columns = ['std', 'p10', 'p25', 'iqr']

    df_filtered = pd.concat([df_filtered, metrics_df], axis=1)

    # Low-Rating Thresholds
    df_filtered['low_rating_by_avg'] = np.where(df_filtered['rating_value'] <= 4, 1, 0)
    df_filtered['low_rating_by_p25'] = np.where(df_filtered['p25'] <= 4, 1, 0)

    print(f"Data ready for plotting. {len(df_filtered)} records.")

    categories = sorted(df_filtered['category'].unique())
    target_states = ['NY', 'CA', 'GA']


    for category in categories:
        print(f"\n>>> Processing Category: {category} <<<")

        for state in target_states:
            plot_data = df_filtered[
                (df_filtered['category'] == category) &
                (df_filtered['State'] == state)
            ]

            if plot_data.empty:
                print(f"No data for {state} in {category}, skipping.")
                continue

            current_order = state_orders[state]
            current_order = [city for city in current_order if city in plot_data['location_name'].unique()]

            # A. Boxplot (Avg, Std)
            fig, axes = plt.subplots(1, 2, figsize=(16, 7))

            # Avg
            sns.boxplot(x='location_name', y='rating_value', data=plot_data, order=current_order, ax=axes[0], palette='Set1', hue='location_name', legend=False)
            axes[0].set_title(f'{state} - Average Rating', fontsize=12)
            axes[0].set_xlabel('Region (SSMLL Order)')
            axes[0].set_ylabel('Rating Value')
            axes[0].tick_params(axis='x', rotation=45)

            # Std
            sns.boxplot(x='location_name', y='std', data=plot_data, order=current_order, ax=axes[1], palette='Set2', hue='location_name', legend=False)
            axes[1].set_title(f'{state} - Standard Deviation', fontsize=12)
            axes[1].set_xlabel('Region (SSMLL Order)')
            axes[1].set_ylabel('Std Dev')
            axes[1].tick_params(axis='x', rotation=45)

            fig.suptitle(get_fig_title(f'{state}: Rating Distribution (Boxplot) for {category}'), fontsize=16)
            plt.tight_layout()
            plt.show()

            # B. Violin Plot (Avg, Std)
            fig, axes = plt.subplots(1, 2, figsize=(16, 7))

            # Avg Violin
            sns.violinplot(x='location_name', y='rating_value', data=plot_data, order=current_order, ax=axes[0], palette='Set1', hue='location_name', legend=False)
            axes[0].set_title(f'{state} - Average Rating', fontsize=12)
            axes[0].tick_params(axis='x', rotation=45)

            # Std Violin
            sns.violinplot(x='location_name', y='std', data=plot_data, order=current_order, ax=axes[1], palette='Set2', hue='location_name', legend=False)
            axes[1].set_title(f'{state} - Standard Deviation', fontsize=12)
            axes[1].tick_params(axis='x', rotation=45)

            fig.suptitle(get_fig_title(f'{state}: Rating Distribution (Violin) for {category}'), fontsize=16)
            plt.tight_layout()
            plt.show()

    # C. Histogram
    print("\n>>> Generating Histograms <<<")
    for category in categories:
        for state in target_states:
            current_order = state_orders[state]
            for location in current_order:
                plot_df = df_filtered[
                    (df_filtered['category'] == category) &
                    (df_filtered['location_name'] == location)
                ]
                if plot_df.empty: continue

                fig, axes = plt.subplots(1, 2, figsize=(14, 5))

                # Avg Hist
                sns.histplot(data=plot_df, x='rating_value', bins=15, kde=True, ax=axes[0], color='blue')
                axes[0].set_title(f'Average Rating', fontsize=11)
                axes[0].set_xlabel('Rating Value')

                # Std Hist
                sns.histplot(data=plot_df, x='std', bins=15, kde=True, ax=axes[1], color='orange')
                axes[1].set_title(f'Standard Deviation', fontsize=11)
                axes[1].set_xlabel('Std Dev')

                fig.suptitle(get_fig_title(f'{state} - {location}: Histograms for {category}'), fontsize=14)
                plt.tight_layout()
                plt.show()

    # D. Low Rating Bar Charts (Aggregated by Type)
    print("\n>>> Low Rating Analysis (Aggregated by Location Type) <<<")

    # Avg Rating <= 4
    low_avg_crosstab = pd.crosstab(df_filtered['location_type'], df_filtered['low_rating_by_avg'])
    rename_dict = {0: 'High Rating (>4)', 1: 'Low Rating (<=4)'}
    low_avg_pct = pd.crosstab(df_filtered['location_type'], df_filtered['low_rating_by_avg'], normalize='index') * 100

    print("Counts:")
    print(low_avg_crosstab.rename(columns=rename_dict))
    print("Percentages: Average Rating")
    print(low_avg_pct.rename(columns=rename_dict).round(2).astype(str) + '%')

    low_avg_crosstab.plot(kind='bar', stacked=True, figsize=(8, 6), color=['skyblue', 'salmon'])
    plt.title(get_fig_title('Count of Low Ratings (Avg <= 4) by Location Type'))
    plt.xlabel('Location Type')
    plt.ylabel('Number of Clinics')
    plt.legend(title='Avg Rating <= 4', labels=['No', 'Yes'])
    plt.tight_layout()
    plt.show()

    # P25 <= 4
    low_p25_crosstab = pd.crosstab(df_filtered['location_type'], df_filtered['low_rating_by_p25'])
    low_p25_pct = pd.crosstab(df_filtered['location_type'], df_filtered['low_rating_by_p25'], normalize='index') * 100

    print("Counts:")
    print(low_p25_crosstab.rename(columns=rename_dict))
    print("Percentages:")
    print(low_p25_pct.rename(columns=rename_dict).round(2).astype(str) + '%')

    low_p25_crosstab.plot(kind='bar', stacked=True, figsize=(8, 6), color=['lightgreen', 'orange'])
    plt.title(get_fig_title('Count of Low Ratings (P25 <= 4) by Location Type'))
    plt.xlabel('Location Type')
    plt.ylabel('Number of Clinics')
    plt.legend(title='25th Percentile <= 4', labels=['No', 'Yes'])
    plt.tight_layout()
    plt.show()

    # E. scatter plots (split by state)
    print("\n>>> Scatter Plots (Split by State) <<<")

    # Scatter
    for state in target_states:
        cities_in_state = state_orders[state]
        cities_in_state = [c for c in cities_in_state if c in df_filtered['location_name'].unique()]

        if not cities_in_state: continue

        n_cities = len(cities_in_state)
        cols = 3
        rows = math.ceil(n_cities / cols)

        fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows), sharex=True, sharey=True)
        if n_cities > 1:
            axes = axes.flatten()
        else:
            axes = [axes]

        for i, city in enumerate(cities_in_state):
            subset = df_filtered[df_filtered['location_name'] == city]

            axes[i].scatter(subset['rating_value'], subset['p25'], alpha=0.6, edgecolors='w', color='blue', label='25th Percentile')
            axes[i].scatter(subset['rating_value'], subset['p10'], alpha=0.6, edgecolors='w', color='orange', label='10th Percentile')

            axes[i].set_title(f"{city} (n={len(subset)})")
            axes[i].grid(True, linestyle='--', alpha=0.5)
            if i == 0:
                axes[i].legend()

        # Labels
        for i, ax in enumerate(axes):
            if i >= (rows - 1) * cols:
                ax.set_xlabel("Average Rating")
            if i % cols == 0:
                ax.set_ylabel("Percentile Rating")
            ax.set_xlim(0.5, 5.5)
            ax.set_ylim(0.5, 5.5)

        if n_cities > 1:
            for j in range(n_cities, len(axes)):
                fig.delaxes(axes[j])

        fig.suptitle(get_fig_title(f"{state}: Scatter Plot (Avg vs P25/P10)"), fontsize=16)
        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        plt.show()

    # Category Scatter
    for category in categories:
        for state in target_states:
            category_df = df_filtered[
                (df_filtered['category'] == category) &
                (df_filtered['State'] == state)
            ]

            cities_in_state = state_orders[state]
            cities_in_state = [c for c in cities_in_state if c in category_df['location_name'].unique()]

            if not cities_in_state: continue

            n_cities = len(cities_in_state)
            cols = 3
            rows = math.ceil(n_cities / cols)

            fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows), sharex=True, sharey=True)
            if n_cities > 1:
                axes = axes.flatten()
            else:
                axes = [axes]

            for i, city in enumerate(cities_in_state):
                subset = category_df[category_df['location_name'] == city]

                axes[i].scatter(subset['rating_value'], subset['p25'], alpha=0.6, edgecolors='w', color='blue', label='P25')
                axes[i].scatter(subset['rating_value'], subset['p10'], alpha=0.6, edgecolors='w', color='orange', label='P10')

                axes[i].set_title(f"{city} (n={len(subset)})")
                axes[i].grid(True, linestyle='--', alpha=0.5)
                if i == 0:
                    axes[i].legend()

            for i, ax in enumerate(axes):
                if i >= (rows - 1) * cols:
                    ax.set_xlabel("Avg Rating")
                if i % cols == 0:
                    ax.set_ylabel("Percentile")
                ax.set_xlim(0.5, 5.5)
                ax.set_ylim(0.5, 5.5)

            if n_cities > 1:
                for j in range(n_cities, len(axes)):
                    fig.delaxes(axes[j])

            fig.suptitle(get_fig_title(f"{state}: Scatter Plot for {category}"), fontsize=16)
            plt.tight_layout(rect=[0, 0.03, 1, 0.95])
            plt.show()

else:
    print("DataFrame is empty. Cannot generate plots.")

In [ ]:
# Constrain By Zipcode
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np
import math

final_csv_path = "/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/output/final/final_processed.csv"


try:
    df = pd.read_csv(final_csv_path)
    print("Successfully loaded final_processed.csv")
except FileNotFoundError:
    print(f"Error: The file '{final_csv_path}' was not found.")
    df = pd.DataFrame()

def get_fig_title(title):
    return title

if not df.empty:
    def clean_zip(val):
        val = str(val).strip()
        if '-' in val:
            val = val.split('-')[0]
        val = ''.join(filter(str.isdigit, val))
        return val[:5]

    df['clean_zip'] = df['zip'].apply(clean_zip)
    df['zip_num'] = pd.to_numeric(df['clean_zip'], errors='coerce')

    def map_zip_to_location(z):
        if pd.isna(z): return 'Unknown'
        z = int(z)

        if z == 12953: return 'Malone_NY_S'
        if z == 12983: return 'SaranacLake_NY_S'
        if 13200 <= z <= 13299: return 'Syracuse_NY_M'
        if 14200 <= z <= 14299: return 'Buffalo_NY_L'
        if (10000 <= z <= 10499) or (11000 <= z <= 11699) or (10300 <= z <= 10399): return 'NYC_NY_L'

        if z in [95501, 95502, 95503, 95534]: return 'Eureka_CA_S'
        if z == 95437: return 'FortBragg_CA_S'
        if 95350 <= z <= 95358: return 'Modesto_CA_M'
        if 94100 <= z <= 94188: return 'SanFrancisco_CA_L'
        if (90000 <= z <= 91699): return 'LA_CA_L'

        if z in [30474, 30475]: return 'Vidalia_GA_S'
        if z == 30577: return 'Toccoa_GA_S'
        if 31200 <= z <= 31299: return 'Macon_GA_M'
        if 30900 <= z <= 30999: return 'Augusta_GA_L'
        if (30300 <= z <= 30399) or (31100 <= z <= 31199): return 'Atlanta_GA_L'

        return 'Unknown'

    df['location_name'] = df['zip_num'].apply(map_zip_to_location)

    df_filtered = df[(df['category'] != 'Unknown') & (df['location_name'] != 'Unknown')].copy()

    state_orders = {
        'NY': ['Malone_NY_S', 'SaranacLake_NY_S', 'Syracuse_NY_M', 'Buffalo_NY_L', 'NYC_NY_L'],
        'CA': ['Eureka_CA_S', 'FortBragg_CA_S', 'Modesto_CA_M', 'SanFrancisco_CA_L', 'LA_CA_L'],
        'GA': ['Vidalia_GA_S', 'Toccoa_GA_S', 'Macon_GA_M', 'Augusta_GA_L', 'Atlanta_GA_L']
    }

    def get_state(loc):
        if '_NY_' in loc: return 'NY'
        if '_CA_' in loc: return 'CA'
        if '_GA_' in loc: return 'GA'
        return 'Other'

    df_filtered['State'] = df_filtered['location_name'].apply(get_state)

    print(f"Records remaining after ZIP filtering: {len(df_filtered)}")

    df_filtered['rating_value'] = pd.to_numeric(df_filtered['rating_value'], errors='coerce')
    df_filtered.dropna(subset=['rating_value', 'category', 'location_name'], inplace=True)

    star_cols = ['rating_1_star', 'rating_2_star', 'rating_3_star', 'rating_4_star', 'rating_5_star']
    for col in star_cols:
        if col in df_filtered.columns:
            df_filtered[col] = pd.to_numeric(df_filtered[col], errors='coerce').fillna(0).astype(int)
        else:
            df_filtered[col] = 0

    def get_location_type(loc):
        if str(loc).endswith('_S'): return 'Small Town/Rural'
        if str(loc).endswith('_M'): return 'Mid-Size City'
        return 'Urban/Metro'

    df_filtered['location_type'] = df_filtered['location_name'].apply(get_location_type)

    def calculate_metrics(row):
        ratings_distribution = []
        try:
            ratings_distribution.extend([1] * row['rating_1_star'])
            ratings_distribution.extend([2] * row['rating_2_star'])
            ratings_distribution.extend([3] * row['rating_3_star'])
            ratings_distribution.extend([4] * row['rating_4_star'])
            ratings_distribution.extend([5] * row['rating_5_star'])
        except:
             return pd.Series([np.nan, np.nan, np.nan, np.nan])

        if not ratings_distribution:
            return pd.Series([np.nan, np.nan, np.nan, np.nan])

        std_dev = np.std(ratings_distribution, ddof=1) if len(ratings_distribution) > 1 else 0
        p10 = np.percentile(ratings_distribution, 10)
        p25 = np.percentile(ratings_distribution, 25)
        p75 = np.percentile(ratings_distribution, 75)
        iqr = p75 - p25

        return pd.Series([std_dev, p10, p25, iqr])

    metrics_df = df_filtered.apply(calculate_metrics, axis=1)
    metrics_df.columns = ['std', 'p10', 'p25', 'iqr']

    df_filtered = pd.concat([df_filtered, metrics_df], axis=1)

    df_filtered['low_rating_by_avg'] = np.where(df_filtered['rating_value'] <= 4, 1, 0)
    df_filtered['low_rating_by_p25'] = np.where(df_filtered['p25'] <= 4, 1, 0)

    categories = sorted(df_filtered['category'].unique())
    target_states = ['NY', 'CA', 'GA']

    for category in categories:
        print(f"\nProcessing Category: {category}")

        for state in target_states:
            plot_data = df_filtered[
                (df_filtered['category'] == category) &
                (df_filtered['State'] == state)
            ]

            if plot_data.empty:
                continue

            current_order = state_orders[state]
            current_order = [city for city in current_order if city in plot_data['location_name'].unique()]

            fig, axes = plt.subplots(1, 2, figsize=(16, 7))

            sns.boxplot(x='location_name', y='rating_value', data=plot_data, order=current_order, ax=axes[0], palette='Set1', hue='location_name', legend=False)
            axes[0].set_title(f'{state} - Average Rating', fontsize=12)
            axes[0].set_xlabel('Region (SSMLL Order)')
            axes[0].set_ylabel('Rating Value')
            axes[0].tick_params(axis='x', rotation=45)

            sns.boxplot(x='location_name', y='std', data=plot_data, order=current_order, ax=axes[1], palette='Set2', hue='location_name', legend=False)
            axes[1].set_title(f'{state} - Standard Deviation', fontsize=12)
            axes[1].set_xlabel('Region (SSMLL Order)')
            axes[1].set_ylabel('Std Dev')
            axes[1].tick_params(axis='x', rotation=45)

            fig.suptitle(get_fig_title(f'{state}: Rating Distribution (Boxplot) for {category}'), fontsize=16)
            plt.tight_layout()
            plt.show()

            fig, axes = plt.subplots(1, 2, figsize=(16, 7))

            sns.violinplot(x='location_name', y='rating_value', data=plot_data, order=current_order, ax=axes[0], palette='Set1', hue='location_name', legend=False)
            axes[0].set_title(f'{state} - Average Rating', fontsize=12)
            axes[0].tick_params(axis='x', rotation=45)

            sns.violinplot(x='location_name', y='std', data=plot_data, order=current_order, ax=axes[1], palette='Set2', hue='location_name', legend=False)
            axes[1].set_title(f'{state} - Standard Deviation', fontsize=12)
            axes[1].tick_params(axis='x', rotation=45)

            fig.suptitle(get_fig_title(f'{state}: Rating Distribution (Violin) for {category}'), fontsize=16)
            plt.tight_layout()
            plt.show()

    print("\nGenerating Histograms")
    for category in categories:
        for state in target_states:
            current_order = state_orders[state]
            for location in current_order:
                plot_df = df_filtered[
                    (df_filtered['category'] == category) &
                    (df_filtered['location_name'] == location)
                ]
                if plot_df.empty: continue

                fig, axes = plt.subplots(1, 2, figsize=(14, 5))

                sns.histplot(data=plot_df, x='rating_value', bins=15, kde=True, ax=axes[0], color='blue')
                axes[0].set_title(f'Average Rating', fontsize=11)
                axes[0].set_xlabel('Rating Value')

                sns.histplot(data=plot_df, x='std', bins=15, kde=True, ax=axes[1], color='orange')
                axes[1].set_title(f'Standard Deviation', fontsize=11)
                axes[1].set_xlabel('Std Dev')

                fig.suptitle(get_fig_title(f'{state} - {location}: Histograms for {category}'), fontsize=14)
                plt.tight_layout()
                plt.show()

    print("\nLow Rating Analysis")

    low_avg_crosstab = pd.crosstab(df_filtered['location_type'], df_filtered['low_rating_by_avg'])
    rename_dict = {0: 'High Rating (>4)', 1: 'Low Rating (<=4)'}
    low_avg_pct = pd.crosstab(df_filtered['location_type'], df_filtered['low_rating_by_avg'], normalize='index') * 100

    print("Counts:")
    print(low_avg_crosstab.rename(columns=rename_dict))
    print("Percentages: Average Rating")
    print(low_avg_pct.rename(columns=rename_dict).round(2).astype(str) + '%')

    low_avg_crosstab.plot(kind='bar', stacked=True, figsize=(8, 6), color=['skyblue', 'salmon'])
    plt.title(get_fig_title('Count of Low Ratings (Avg <= 4) by Location Type'))
    plt.xlabel('Location Type')
    plt.ylabel('Number of Clinics')
    plt.legend(title='Avg Rating <= 4', labels=['No', 'Yes'])
    plt.tight_layout()
    plt.show()

    low_p25_crosstab = pd.crosstab(df_filtered['location_type'], df_filtered['low_rating_by_p25'])
    low_p25_pct = pd.crosstab(df_filtered['location_type'], df_filtered['low_rating_by_p25'], normalize='index') * 100

    print("Counts:")
    print(low_p25_crosstab.rename(columns=rename_dict))
    print("Percentages:")
    print(low_p25_pct.rename(columns=rename_dict).round(2).astype(str) + '%')

    low_p25_crosstab.plot(kind='bar', stacked=True, figsize=(8, 6), color=['lightgreen', 'orange'])
    plt.title(get_fig_title('Count of Low Ratings (P25 <= 4) by Location Type'))
    plt.xlabel('Location Type')
    plt.ylabel('Number of Clinics')
    plt.legend(title='25th Percentile <= 4', labels=['No', 'Yes'])
    plt.tight_layout()
    plt.show()

    print("\nScatter Plots (Split by State)")

    for state in target_states:
        cities_in_state = state_orders[state]
        cities_in_state = [c for c in cities_in_state if c in df_filtered['location_name'].unique()]

        if not cities_in_state: continue

        n_cities = len(cities_in_state)
        cols = 3
        rows = math.ceil(n_cities / cols)

        fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows), sharex=True, sharey=True)
        if n_cities > 1:
            axes = axes.flatten()
        else:
            axes = [axes]

        for i, city in enumerate(cities_in_state):
            subset = df_filtered[df_filtered['location_name'] == city]

            axes[i].scatter(subset['rating_value'], subset['p25'], alpha=0.6, edgecolors='w', color='blue', label='25th Percentile')
            axes[i].scatter(subset['rating_value'], subset['p10'], alpha=0.6, edgecolors='w', color='orange', label='10th Percentile')

            axes[i].set_title(f"{city} (n={len(subset)})")
            axes[i].grid(True, linestyle='--', alpha=0.5)
            if i == 0:
                axes[i].legend()

        for i, ax in enumerate(axes):
            if i >= (rows - 1) * cols:
                ax.set_xlabel("Average Rating")
            if i % cols == 0:
                ax.set_ylabel("Percentile Rating")
            ax.set_xlim(0.5, 5.5)
            ax.set_ylim(0.5, 5.5)

        if n_cities > 1:
            for j in range(n_cities, len(axes)):
                fig.delaxes(axes[j])

        fig.suptitle(get_fig_title(f"{state}: Scatter Plot (Avg vs P25/P10)"), fontsize=16)
        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        plt.show()

    for category in categories:
        for state in target_states:
            category_df = df_filtered[
                (df_filtered['category'] == category) &
                (df_filtered['State'] == state)
            ]

            cities_in_state = state_orders[state]
            cities_in_state = [c for c in cities_in_state if c in category_df['location_name'].unique()]

            if not cities_in_state: continue

            n_cities = len(cities_in_state)
            cols = 3
            rows = math.ceil(n_cities / cols)

            fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows), sharex=True, sharey=True)
            if n_cities > 1:
                axes = axes.flatten()
            else:
                axes = [axes]

            for i, city in enumerate(cities_in_state):
                subset = category_df[category_df['location_name'] == city]

                axes[i].scatter(subset['rating_value'], subset['p25'], alpha=0.6, edgecolors='w', color='blue', label='P25')
                axes[i].scatter(subset['rating_value'], subset['p10'], alpha=0.6, edgecolors='w', color='orange', label='P10')

                axes[i].set_title(f"{city} (n={len(subset)})")
                axes[i].grid(True, linestyle='--', alpha=0.5)
                if i == 0:
                    axes[i].legend()

            for i, ax in enumerate(axes):
                if i >= (rows - 1) * cols:
                    ax.set_xlabel("Avg Rating")
                if i % cols == 0:
                    ax.set_ylabel("Percentile")
                ax.set_xlim(0.5, 5.5)
                ax.set_ylim(0.5, 5.5)

            if n_cities > 1:
                for j in range(n_cities, len(axes)):
                    fig.delaxes(axes[j])

            fig.suptitle(get_fig_title(f"{state}: Scatter Plot for {category}"), fontsize=16)
            plt.tight_layout(rect=[0, 0.03, 1, 0.95])
            plt.show()

else:
    print("DataFrame is empty.")

# New one plot

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np
import math

# Load Data
final_csv_path = "/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/output/final/final_processed.csv"

print(f"Loading data from: {final_csv_path}")

try:
    df = pd.read_csv(final_csv_path)
    print("Successfully loaded final_processed.csv")
except FileNotFoundError:
    print(f"Error: The file '{final_csv_path}' was not found.")
    df = pd.DataFrame()

def get_fig_title(title):
    return title

if not df.empty:
    # Preprocess
    if 'search_location' not in df.columns:
        if 'location_name' in df.columns:
            df['search_location'] = df['location_name']
        else:
            print("Error: Missing location columns.")
            df = pd.DataFrame()

if not df.empty:
    # Define mapping
    state_orders = {
        'NY': ['Malone_NY_S', 'SaranacLake_NY_S', 'Syracuse_NY_M', 'Buffalo_NY_L', 'NYC_NY_L'],
        'CA': ['Eureka_CA_S', 'FortBragg_CA_S', 'Modesto_CA_M', 'SanFrancisco_CA_L', 'LA_CA_L'],
        'GA': ['Vidalia_GA_S', 'Toccoa_GA_S', 'Macon_GA_M', 'Augusta_GA_L', 'Atlanta_GA_L']
    }
    all_target_cities = [city for cities in state_orders.values() for city in cities]

    # Clean & Filter
    df['search_location'] = df['search_location'].astype(str)
    df_filtered = df[
        (df['category'] != 'Unknown') &
        (df['search_location'].isin(all_target_cities))
    ].copy()
    df_filtered['location_name'] = df_filtered['search_location']

    # Convert ratings
    df_filtered['rating_value'] = pd.to_numeric(df_filtered['rating_value'], errors='coerce')
    df_filtered.dropna(subset=['rating_value', 'category', 'location_name'], inplace=True)

    # Process stars
    star_cols = ['rating_1_star', 'rating_2_star', 'rating_3_star', 'rating_4_star', 'rating_5_star']
    for col in star_cols:
        if col in df_filtered.columns:
            df_filtered[col] = pd.to_numeric(df_filtered[col], errors='coerce').fillna(0).astype(int)
        else:
            df_filtered[col] = 0

    # Define location size type
    def get_location_info(loc):
        s_loc = str(loc)
        if s_loc.endswith('_S'):
            return 'Small'
        elif s_loc.endswith('_M'):
            return 'Mid-Size'
        elif s_loc.endswith('_L'):
            return 'Large'
        return 'Unknown'

    df_filtered['Location_Type'] = df_filtered['location_name'].apply(get_location_info)

    # Sort by population
    master_locations_order = [
        # Small
        'SaranacLake_NY_S',
        'FortBragg_CA_S',
        'Toccoa_GA_S',
        'Vidalia_GA_S',
        'Malone_NY_S',
        'Eureka_CA_S',

        # Mid-Size
        'Macon_GA_M',
        'Syracuse_NY_M',
        'Modesto_CA_M',

        # Large
        'Augusta_GA_L',
        'Buffalo_NY_L',
        'Atlanta_GA_L',
        'SanFrancisco_CA_L',
        'LA_CA_L',
        'NYC_NY_L'
    ]

    print("Sorted City Order:")
    print(master_locations_order)

    # Calculate Metrics
    def calculate_metrics(row):
        ratings_distribution = []
        try:
            ratings_distribution.extend([1] * row['rating_1_star'])
            ratings_distribution.extend([2] * row['rating_2_star'])
            ratings_distribution.extend([3] * row['rating_3_star'])
            ratings_distribution.extend([4] * row['rating_4_star'])
            ratings_distribution.extend([5] * row['rating_5_star'])
        except:
             return pd.Series([np.nan, np.nan, np.nan, np.nan])

        if not ratings_distribution:
            return pd.Series([np.nan, np.nan, np.nan, np.nan])

        std_dev = np.std(ratings_distribution, ddof=1) if len(ratings_distribution) > 1 else 0
        p10 = np.percentile(ratings_distribution, 10)
        p25 = np.percentile(ratings_distribution, 25)
        p75 = np.percentile(ratings_distribution, 75)
        iqr = p75 - p25
        return pd.Series([std_dev, p10, p25, iqr])

    metrics_df = df_filtered.apply(calculate_metrics, axis=1)
    metrics_df.columns = ['std', 'p10', 'p25', 'iqr']
    df_filtered = pd.concat([df_filtered, metrics_df], axis=1)

    # Low rating
    df_filtered['low_rating_by_avg'] = np.where(df_filtered['rating_value'] <= 3, 1, 0)
    df_filtered['low_rating_by_p25'] = np.where(df_filtered['p25'] <= 3, 1, 0)

    categories = sorted(df_filtered['category'].unique())

    # Small, mid, large: blue, orange, green
    custom_colors = ['tab:blue', 'tab:orange', 'tab:green']

    # plot
    for category in categories:
        print(f"Category: {category}")
        cat_data = df_filtered[df_filtered['category'] == category]
        if cat_data.empty: continue

        current_order = [city for city in master_locations_order if city in cat_data['location_name'].unique()]

        # 1. Boxplots
        fig, axes = plt.subplots(1, 2, figsize=(24, 8))
        hue_order = ['Small', 'Mid-Size', 'Large']

        sns.boxplot(x='location_name', y='rating_value', data=cat_data, order=current_order, ax=axes[0],
                    hue='Location_Type', hue_order=hue_order, palette=custom_colors, dodge=False)
        axes[0].set_title(f'Average Rating by City (Sorted by Population)', fontsize=14)
        axes[0].tick_params(axis='x', rotation=45)
        axes[0].set_xlabel("City (Small -> Large)")

        sns.boxplot(x='location_name', y='std', data=cat_data, order=current_order, ax=axes[1],
                    hue='Location_Type', hue_order=hue_order, palette=custom_colors, dodge=False)
        axes[1].set_title(f'Standard Deviation by City (Sorted by Population)', fontsize=14)
        axes[1].tick_params(axis='x', rotation=45)
        axes[1].set_xlabel("City (Small -> Large)")

        plt.tight_layout()
        plt.show()

        # 2. Violin
        fig, axes = plt.subplots(1, 2, figsize=(24, 8))
        sns.violinplot(x='location_name', y='rating_value', data=cat_data, order=current_order, ax=axes[0],
                       hue='Location_Type', hue_order=hue_order, palette=custom_colors, dodge=False)
        axes[0].set_title(f'Avg Rating Distribution', fontsize=14)
        axes[0].tick_params(axis='x', rotation=45)

        sns.violinplot(x='location_name', y='std', data=cat_data, order=current_order, ax=axes[1],
                       hue='Location_Type', hue_order=hue_order, palette=custom_colors, dodge=False)
        axes[1].set_title(f'Std Dev Distribution', fontsize=14)
        axes[1].tick_params(axis='x', rotation=45)
        plt.tight_layout()
        plt.show()

        # 3. Histograms
        print("Aggregated Histograms")
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))

        # Avg Rating Hist
        sns.histplot(data=cat_data, x='rating_value', hue='Location_Type', bins=10, kde=True,
                     element="step", stat="density", common_norm=False, ax=axes[0],
                     hue_order=hue_order, palette=custom_colors)
        axes[0].set_title('Average Rating Distribution by Size')

        # Std Dev Hist
        sns.histplot(data=cat_data, x='std', hue='Location_Type', bins=10, kde=True,
                     element="step", stat="density", common_norm=False, ax=axes[1],
                     hue_order=hue_order, palette=custom_colors)
        axes[1].set_title('Standard Deviation Distribution by Size')

        plt.tight_layout()
        plt.show()

        # 4. Scatter Plots
        print("Aggregated Scatter Plot")
        fig, ax = plt.subplots(figsize=(10, 7))

        # Scatter: X=Avg, Y=P25, Hue=Type
        sns.scatterplot(data=cat_data, x='rating_value', y='p25', hue='Location_Type', style='Location_Type',
                        hue_order=hue_order, style_order=hue_order,
                        palette=custom_colors, s=80, alpha=0.7, ax=ax)

        ax.set_title(f'Scatter: Avg Rating vs 25th Percentile ({category})')
        ax.set_xlabel('Average Rating')
        ax.set_ylabel('25th Percentile Rating')
        ax.grid(True, linestyle='--', alpha=0.3)
        plt.tight_layout()
        plt.show()

    # Low rating analysis
    print("Low Rating Analysis")

    hue_order_bar = ['Small', 'Mid-Size', 'Large']
    rename_dict = {0: 'High (>3)', 1: 'Low (<=3)'}

    # Avg <= 3
    low_avg_pct = pd.crosstab(df_filtered['Location_Type'], df_filtered['low_rating_by_avg'], normalize='index').reindex(hue_order_bar) * 100
    print("Percentages (Avg Rating <= 3)")
    print(low_avg_pct.rename(columns=rename_dict).round(2).astype(str) + '%')

    pd.crosstab(df_filtered['Location_Type'], df_filtered['low_rating_by_avg']).reindex(hue_order_bar).rename(columns=rename_dict).plot(
        kind='bar', stacked=True, figsize=(8, 6), color=['skyblue', 'salmon'])
    plt.title('Low Rating (Avg <= 3) Counts by Type')
    plt.ylabel('Count')
    plt.xticks(rotation=0)
    plt.show()

    # P25 <= 3
    low_p25_pct = pd.crosstab(df_filtered['Location_Type'], df_filtered['low_rating_by_p25'], normalize='index').reindex(hue_order_bar) * 100
    print("Percentages (25th Percentile <= 3)")
    print(low_p25_pct.rename(columns=rename_dict).round(2).astype(str) + '%')

    pd.crosstab(df_filtered['Location_Type'], df_filtered['low_rating_by_p25']).reindex(hue_order_bar).rename(columns=rename_dict).plot(
        kind='bar', stacked=True, figsize=(8, 6), color=['lightgreen', 'orange'])
    plt.title('Low Rating (P25 <= 3) Counts by Type')
    plt.ylabel('Count')
    plt.xticks(rotation=0)
    plt.show()
else:
    print("DataFrame is empty.")

# One Plot only avg

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np
import math

# Load Data
final_csv_path = "/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/output/final/final_processed.csv"

print(f"Loading data from: {final_csv_path}")

try:
    df = pd.read_csv(final_csv_path)
    print("Successfully loaded final_processed.csv")
except FileNotFoundError:
    print(f"Error: The file '{final_csv_path}' was not found.")
    df = pd.DataFrame()

def get_fig_title(title):
    return title

if not df.empty:
    # Preprocess
    if 'search_location' not in df.columns:
        if 'location_name' in df.columns:
            df['search_location'] = df['location_name']
        else:
            print("Error: Missing location columns.")
            df = pd.DataFrame()

if not df.empty:
    # Define mapping
    state_orders = {
        'NY': ['Malone_NY_S', 'SaranacLake_NY_S', 'Syracuse_NY_M', 'Buffalo_NY_L', 'NYC_NY_L'],
        'CA': ['Eureka_CA_S', 'FortBragg_CA_S', 'Modesto_CA_M', 'SanFrancisco_CA_L', 'LA_CA_L'],
        'GA': ['Vidalia_GA_S', 'Toccoa_GA_S', 'Macon_GA_M', 'Augusta_GA_L', 'Atlanta_GA_L']
    }
    all_target_cities = [city for cities in state_orders.values() for city in cities]

    # Clean & Filter
    df['search_location'] = df['search_location'].astype(str)
    df_filtered = df[
        (df['category'] != 'Unknown') &
        (df['search_location'].isin(all_target_cities))
    ].copy()
    df_filtered['location_name'] = df_filtered['search_location']

    # Convert ratings
    df_filtered['rating_value'] = pd.to_numeric(df_filtered['rating_value'], errors='coerce')
    df_filtered.dropna(subset=['rating_value', 'category', 'location_name'], inplace=True)

    # Process stars
    star_cols = ['rating_1_star', 'rating_2_star', 'rating_3_star', 'rating_4_star', 'rating_5_star']
    for col in star_cols:
        if col in df_filtered.columns:
            df_filtered[col] = pd.to_numeric(df_filtered[col], errors='coerce').fillna(0).astype(int)
        else:
            df_filtered[col] = 0

    # Define location size type
    def get_location_info(loc):
        s_loc = str(loc)
        if s_loc.endswith('_S'):
            return 'Small'
        elif s_loc.endswith('_M'):
            return 'Mid-Size'
        elif s_loc.endswith('_L'):
            return 'Large'
        return 'Unknown'

    df_filtered['Location_Type'] = df_filtered['location_name'].apply(get_location_info)

    # Sort by population
    master_locations_order = [
        # Small
        'SaranacLake_NY_S',
        'FortBragg_CA_S',
        'Toccoa_GA_S',
        'Vidalia_GA_S',
        'Malone_NY_S',
        'Eureka_CA_S',

        # Mid-Size
        'Macon_GA_M',
        'Syracuse_NY_M',
        'Modesto_CA_M',

        # Large
        'Augusta_GA_L',
        'Buffalo_NY_L',
        'Atlanta_GA_L',
        'SanFrancisco_CA_L',
        'LA_CA_L',
        'NYC_NY_L'
    ]

    print("Sorted City Order:")
    print(master_locations_order)

    # Calculate Metrics
    def calculate_metrics(row):
        ratings_distribution = []
        try:
            ratings_distribution.extend([1] * row['rating_1_star'])
            ratings_distribution.extend([2] * row['rating_2_star'])
            ratings_distribution.extend([3] * row['rating_3_star'])
            ratings_distribution.extend([4] * row['rating_4_star'])
            ratings_distribution.extend([5] * row['rating_5_star'])
        except:
             return pd.Series([np.nan, np.nan, np.nan])

        if not ratings_distribution:
            return pd.Series([np.nan, np.nan, np.nan])

        p10 = np.percentile(ratings_distribution, 10)
        p25 = np.percentile(ratings_distribution, 25)
        p75 = np.percentile(ratings_distribution, 75)
        iqr = p75 - p25
        return pd.Series([p10, p25, iqr])

    metrics_df = df_filtered.apply(calculate_metrics, axis=1)
    metrics_df.columns = ['p10', 'p25', 'iqr']
    df_filtered = pd.concat([df_filtered, metrics_df], axis=1)

    # Low rating
    df_filtered['low_rating_by_avg'] = np.where(df_filtered['rating_value'] <= 3, 1, 0)
    df_filtered['low_rating_by_p25'] = np.where(df_filtered['p25'] <= 3, 1, 0)

    categories = sorted(df_filtered['category'].unique())

    # Small, mid, large: blue, orange, green
    custom_colors = ['tab:blue', 'tab:orange', 'tab:green']

    # plot
    for category in categories:
        print(f"Category: {category}")
        cat_data = df_filtered[df_filtered['category'] == category]
        if cat_data.empty: continue

        current_order = [city for city in master_locations_order if city in cat_data['location_name'].unique()]

        # 1. Boxplots
        fig, ax = plt.subplots(figsize=(12, 8))
        hue_order = ['Small', 'Mid-Size', 'Large']

        sns.boxplot(x='location_name', y='rating_value', data=cat_data, order=current_order, ax=ax,
                    hue='Location_Type', hue_order=hue_order, palette=custom_colors, dodge=False)
        ax.set_title(f'Average Rating by City (Sorted by Population)', fontsize=14)
        ax.tick_params(axis='x', rotation=45)
        ax.set_xlabel("City (Small -> Large)")

        plt.tight_layout()
        plt.show()

        # 2. Violin
        fig, ax = plt.subplots(figsize=(12, 8))
        sns.violinplot(x='location_name', y='rating_value', data=cat_data, order=current_order, ax=ax,
                       hue='Location_Type', hue_order=hue_order, palette=custom_colors, dodge=False)
        ax.set_title(f'Avg Rating Distribution', fontsize=14)
        ax.tick_params(axis='x', rotation=45)

        plt.tight_layout()
        plt.show()

        # 3. Histograms
        print("Aggregated Histograms")
        fig, ax = plt.subplots(figsize=(8, 6))

        # Avg Rating Hist
        sns.histplot(data=cat_data, x='rating_value', hue='Location_Type', bins=10, kde=True,
                     element="step", stat="density", common_norm=False, ax=ax,
                     hue_order=hue_order, palette=custom_colors)
        ax.set_title('Average Rating Distribution by Size')

        plt.tight_layout()
        plt.show()

        # 4. Scatter Plots
        print("Aggregated Scatter Plot")
        fig, ax = plt.subplots(figsize=(10, 7))

        # Scatter: X=Avg, Y=P25, Hue=Type
        sns.scatterplot(data=cat_data, x='rating_value', y='p25', hue='Location_Type', style='Location_Type',
                        hue_order=hue_order, style_order=hue_order,
                        palette=custom_colors, s=80, alpha=0.7, ax=ax)

        ax.set_title(f'Scatter: Avg Rating vs 25th Percentile ({category})')
        ax.set_xlabel('Average Rating')
        ax.set_ylabel('25th Percentile Rating')
        ax.grid(True, linestyle='--', alpha=0.3)
        plt.tight_layout()
        plt.show()

    # Low rating analysis
    print("Low Rating Analysis")

    hue_order_bar = ['Small', 'Mid-Size', 'Large']
    rename_dict = {0: 'High (>3)', 1: 'Low (<=3)'}

    # Avg <= 3
    low_avg_pct = pd.crosstab(df_filtered['Location_Type'], df_filtered['low_rating_by_avg'], normalize='index').reindex(hue_order_bar) * 100
    print("Percentages (Avg Rating <= 3)")
    print(low_avg_pct.rename(columns=rename_dict).round(2).astype(str) + '%')

    pd.crosstab(df_filtered['Location_Type'], df_filtered['low_rating_by_avg']).reindex(hue_order_bar).rename(columns=rename_dict).plot(
        kind='bar', stacked=True, figsize=(8, 6), color=['skyblue', 'salmon'])
    plt.title('Low Rating (Avg <= 3) Counts by Type')
    plt.ylabel('Count')
    plt.xticks(rotation=0)
    plt.show()

    # P25 <= 3
    low_p25_pct = pd.crosstab(df_filtered['Location_Type'], df_filtered['low_rating_by_p25'], normalize='index').reindex(hue_order_bar) * 100
    print("Percentages (25th Percentile <= 3)")
    print(low_p25_pct.rename(columns=rename_dict).round(2).astype(str) + '%')

    pd.crosstab(df_filtered['Location_Type'], df_filtered['low_rating_by_p25']).reindex(hue_order_bar).rename(columns=rename_dict).plot(
        kind='bar', stacked=True, figsize=(8, 6), color=['lightgreen', 'orange'])
    plt.title('Low Rating (P25 <= 3) Counts by Type')
    plt.ylabel('Count')
    plt.xticks(rotation=0)
    plt.show()
else:
    print("DataFrame is empty.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np
import math

# Load Data
final_csv_path = "/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/output/final/final_processed.csv"

print(f"Loading data from: {final_csv_path}")

try:
    df = pd.read_csv(final_csv_path)
    print("Successfully loaded final_processed.csv")
except FileNotFoundError:
    print(f"Error: The file '{final_csv_path}' was not found.")
    df = pd.DataFrame()

def get_fig_title(title):
    return title

if not df.empty:
    # Clean zip
    def clean_zip_code(val):
        val = str(val).strip()
        if '-' in val:
            val = val.split('-')[0]
        # Keep only digits, and 5 first digit
        val = ''.join(filter(str.isdigit, val))

        val = val[:5]

        if val:
            return float(val)
        return np.nan

    if 'zip' in df.columns:
        df['zip_clean'] = df['zip'].apply(clean_zip_code)
    else:
        print("Warning: 'zip' column not found. Skipping ZIP filtering.")
        df['zip_clean'] = np.nan

    # zip mapping
    def map_zip_to_location(z):
        if pd.isna(z): return 'Unknown'
        try:
            z = int(z)
        except ValueError:
            return 'Unknown'

        if z == 12953: return 'Malone_NY_S'
        if z == 12983: return 'SaranacLake_NY_S'
        if 13200 <= z <= 13299: return 'Syracuse_NY_M'
        if 14200 <= z <= 14299: return 'Buffalo_NY_L'
        if (10000 <= z <= 10499) or (11000 <= z <= 11699) or (10300 <= z <= 10399): return 'NYC_NY_L'

        if z in [95501, 95502, 95503, 95534]: return 'Eureka_CA_S'
        if z == 95437: return 'FortBragg_CA_S'
        if 95350 <= z <= 95358: return 'Modesto_CA_M'
        if 94100 <= z <= 94188: return 'SanFrancisco_CA_L'
        if (90000 <= z <= 91699): return 'LA_CA_L'

        if z in [30474, 30475]: return 'Vidalia_GA_S'
        if z == 30577: return 'Toccoa_GA_S'
        if 31200 <= z <= 31299: return 'Macon_GA_M'
        if 30900 <= z <= 30999: return 'Augusta_GA_L'
        if (30300 <= z <= 30399) or (31100 <= z <= 31199): return 'Atlanta_GA_L'

        return 'Unknown'

    df['mapped_location'] = df['zip_clean'].apply(map_zip_to_location)

    print(f"Records before filtering: {len(df)}")

    # Clean up Unknowns
    df = df[df['mapped_location'] != 'Unknown'].copy()

    # Update location columns
    df['search_location'] = df['mapped_location']
    df['location_name'] = df['mapped_location']

    print(f"Records remaining after zipcode filtering: {len(df)}")


    # Define mapping
    state_orders = {
        'NY': ['Malone_NY_S', 'SaranacLake_NY_S', 'Syracuse_NY_M', 'Buffalo_NY_L', 'NYC_NY_L'],
        'CA': ['Eureka_CA_S', 'FortBragg_CA_S', 'Modesto_CA_M', 'SanFrancisco_CA_L', 'LA_CA_L'],
        'GA': ['Vidalia_GA_S', 'Toccoa_GA_S', 'Macon_GA_M', 'Augusta_GA_L', 'Atlanta_GA_L']
    }
    all_target_cities = [city for cities in state_orders.values() for city in cities]

    # Clean & Filter
    df['search_location'] = df['search_location'].astype(str)
    df_filtered = df[
        (df['category'] != 'Unknown') &
        (df['search_location'].isin(all_target_cities))
    ].copy()
    df_filtered['location_name'] = df_filtered['search_location']

    # Convert ratings
    df_filtered['rating_value'] = pd.to_numeric(df_filtered['rating_value'], errors='coerce')
    df_filtered.dropna(subset=['rating_value', 'category', 'location_name'], inplace=True)

    # Process stars
    star_cols = ['rating_1_star', 'rating_2_star', 'rating_3_star', 'rating_4_star', 'rating_5_star']
    for col in star_cols:
        if col in df_filtered.columns:
            df_filtered[col] = pd.to_numeric(df_filtered[col], errors='coerce').fillna(0).astype(int)
        else:
            df_filtered[col] = 0

    # Define location size type
    def get_location_info(loc):
        s_loc = str(loc)
        if s_loc.endswith('_S'):
            return 'Small'
        elif s_loc.endswith('_M'):
            return 'Mid-Size'
        elif s_loc.endswith('_L'):
            return 'Large'
        return 'Unknown'

    df_filtered['Location_Type'] = df_filtered['location_name'].apply(get_location_info)

    # Sort by population
    master_locations_order = [
        # Small
        'SaranacLake_NY_S',
        'FortBragg_CA_S',
        'Toccoa_GA_S',
        'Vidalia_GA_S',
        'Malone_NY_S',
        'Eureka_CA_S',

        # Mid-Size
        'Macon_GA_M',
        'Syracuse_NY_M',
        'Modesto_CA_M',

        # Large
        'Augusta_GA_L',
        'Buffalo_NY_L',
        'Atlanta_GA_L',
        'SanFrancisco_CA_L',
        'LA_CA_L',
        'NYC_NY_L'
    ]

    print("Sorted City Order:")
    print(master_locations_order)

    # Calculate Metrics
    def calculate_metrics(row):
        ratings_distribution = []
        try:
            ratings_distribution.extend([1] * row['rating_1_star'])
            ratings_distribution.extend([2] * row['rating_2_star'])
            ratings_distribution.extend([3] * row['rating_3_star'])
            ratings_distribution.extend([4] * row['rating_4_star'])
            ratings_distribution.extend([5] * row['rating_5_star'])
        except:
             return pd.Series([np.nan, np.nan, np.nan])

        if not ratings_distribution:
            return pd.Series([np.nan, np.nan, np.nan])

        p10 = np.percentile(ratings_distribution, 10)
        p25 = np.percentile(ratings_distribution, 25)
        p75 = np.percentile(ratings_distribution, 75)
        iqr = p75 - p25
        return pd.Series([p10, p25, iqr])

    metrics_df = df_filtered.apply(calculate_metrics, axis=1)
    metrics_df.columns = ['p10', 'p25', 'iqr']
    df_filtered = pd.concat([df_filtered, metrics_df], axis=1)

    # Low rating
    df_filtered['low_rating_by_avg'] = np.where(df_filtered['rating_value'] <= 3, 1, 0)
    df_filtered['low_rating_by_p25'] = np.where(df_filtered['p25'] <= 3, 1, 0)

    categories = sorted(df_filtered['category'].unique())

    # Small, mid, large: blue, orange, green
    custom_colors = ['tab:blue', 'tab:orange', 'tab:green']

    # plot
    for category in categories:
        print(f"Category: {category}")
        cat_data = df_filtered[df_filtered['category'] == category]
        if cat_data.empty: continue

        current_order = [city for city in master_locations_order if city in cat_data['location_name'].unique()]

        # 1. Boxplots
        fig, ax = plt.subplots(figsize=(12, 8))
        hue_order = ['Small', 'Mid-Size', 'Large']

        sns.boxplot(x='location_name', y='rating_value', data=cat_data, order=current_order, ax=ax,
                    hue='Location_Type', hue_order=hue_order, palette=custom_colors, dodge=False)
        ax.set_title(f'Average Rating by City (Sorted by Population)', fontsize=14)
        ax.tick_params(axis='x', rotation=45)
        ax.set_xlabel("City (Small -> Large)")

        plt.tight_layout()
        plt.show()

        # 2. Violin
        fig, ax = plt.subplots(figsize=(12, 8))
        sns.violinplot(x='location_name', y='rating_value', data=cat_data, order=current_order, ax=ax,
                       hue='Location_Type', hue_order=hue_order, palette=custom_colors, dodge=False)
        ax.set_title(f'Avg Rating Distribution', fontsize=14)
        ax.tick_params(axis='x', rotation=45)

        plt.tight_layout()
        plt.show()

        # 3. Histograms
        print("Aggregated Histograms")
        fig, ax = plt.subplots(figsize=(8, 6))

        # Avg Rating Hist
        sns.histplot(data=cat_data, x='rating_value', hue='Location_Type', bins=10, kde=True,
                     element="step", stat="density", common_norm=False, ax=ax,
                     hue_order=hue_order, palette=custom_colors)
        ax.set_title('Average Rating Distribution by Size')

        plt.tight_layout()
        plt.show()

        # 4. Scatter Plots
        print("Aggregated Scatter Plot")
        fig, ax = plt.subplots(figsize=(10, 7))

        # Scatter: X=Avg, Y=P25, Hue=Type
        sns.scatterplot(data=cat_data, x='rating_value', y='p25', hue='Location_Type', style='Location_Type',
                        hue_order=hue_order, style_order=hue_order,
                        palette=custom_colors, s=80, alpha=0.7, ax=ax)

        ax.set_title(f'Scatter: Avg Rating vs 25th Percentile ({category})')
        ax.set_xlabel('Average Rating')
        ax.set_ylabel('25th Percentile Rating')
        ax.grid(True, linestyle='--', alpha=0.3)
        plt.tight_layout()
        plt.show()

    # Low rating analysis
    print("Low Rating Analysis")

    hue_order_bar = ['Small', 'Mid-Size', 'Large']
    rename_dict = {0: 'High (>3)', 1: 'Low (<=3)'}

    # Avg <= 3
    low_avg_pct = pd.crosstab(df_filtered['Location_Type'], df_filtered['low_rating_by_avg'], normalize='index').reindex(hue_order_bar) * 100
    print("Percentages (Avg Rating <= 3)")
    print(low_avg_pct.rename(columns=rename_dict).round(2).astype(str) + '%')

    pd.crosstab(df_filtered['Location_Type'], df_filtered['low_rating_by_avg']).reindex(hue_order_bar).rename(columns=rename_dict).plot(
        kind='bar', stacked=True, figsize=(8, 6), color=['skyblue', 'salmon'])
    plt.title('Low Rating (Avg <= 3) Counts by Type')
    plt.ylabel('Count')
    plt.xticks(rotation=0)
    plt.show()

    # P25 <= 3
    low_p25_pct = pd.crosstab(df_filtered['Location_Type'], df_filtered['low_rating_by_p25'], normalize='index').reindex(hue_order_bar) * 100
    print("Percentages (25th Percentile <= 3)")
    print(low_p25_pct.rename(columns=rename_dict).round(2).astype(str) + '%')

    pd.crosstab(df_filtered['Location_Type'], df_filtered['low_rating_by_p25']).reindex(hue_order_bar).rename(columns=rename_dict).plot(
        kind='bar', stacked=True, figsize=(8, 6), color=['lightgreen', 'orange'])
    plt.title('Low Rating (P25 <= 3) Counts by Type')
    plt.ylabel('Count')
    plt.xticks(rotation=0)
    plt.show()
else:
    print("DataFrame is empty.")

# Regression

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm
from math import radians, cos, sin, asin, sqrt

# Load Data
final_csv_path = "/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/output/final/final_processed.csv"

try:
    df = pd.read_csv(final_csv_path)
    print("Successfully loaded data.")
except FileNotFoundError:
    print("File not found.")
    df = pd.DataFrame()

if not df.empty:
    # map zipcode
    def clean_zip_code(val):
        val = str(val).strip()
        if '-' in val: val = val.split('-')[0]
        val = ''.join(filter(str.isdigit, val))
        val = val[:5]
        if val: return float(val)
        return np.nan

    if 'zip' in df.columns:
        df['zip_clean'] = df['zip'].apply(clean_zip_code)
    else:
        df['zip_clean'] = np.nan

    def map_zip_to_location(z):
        if pd.isna(z): return 'Unknown'
        try: z = int(z)
        except ValueError: return 'Unknown'
        if z == 12953: return 'Malone_NY_S'
        if z == 12983: return 'SaranacLake_NY_S'
        if 13200 <= z <= 13299: return 'Syracuse_NY_M'
        if 14200 <= z <= 14299: return 'Buffalo_NY_L'
        if (10000 <= z <= 10499) or (11000 <= z <= 11699) or (10300 <= z <= 10399): return 'NYC_NY_L'
        if z in [95501, 95502, 95503, 95534]: return 'Eureka_CA_S'
        if z == 95437: return 'FortBragg_CA_S'
        if 95350 <= z <= 95358: return 'Modesto_CA_M'
        if 94100 <= z <= 94188: return 'SanFrancisco_CA_L'
        if (90000 <= z <= 91699): return 'LA_CA_L'
        if z in [30474, 30475]: return 'Vidalia_GA_S'
        if z == 30577: return 'Toccoa_GA_S'
        if 31200 <= z <= 31299: return 'Macon_GA_M'
        if 30900 <= z <= 30999: return 'Augusta_GA_L'
        if (30300 <= z <= 30399) or (31100 <= z <= 31199): return 'Atlanta_GA_L'
        return 'Unknown'

    df['mapped_location'] = df['zip_clean'].apply(map_zip_to_location)
    df = df[df['mapped_location'] != 'Unknown'].copy()

    # Category (general, special, urgent/surgery)
    cat_mapping = {
        'keywords_General_Dentist': 'General',
        'keywords_Special_Dentist': 'Special',
        'keywords_Surgery_Dentist': 'Surgery'
    }

    if 'category' in df.columns:
        df = df[df['category'].isin(cat_mapping.keys())].copy()
        df['Keyword_Category'] = df['category'].map(cat_mapping)
        print(df['Keyword_Category'].value_counts())
    else:
        print("Error: 'category' column missing")
        df = pd.DataFrame()

    df['Keyword_Category'] = pd.Categorical(
        df['Keyword_Category'],
        categories=['General', 'Special', 'Surgery'],
        ordered=False
    )

    # Location Size (Small/Mid/Large)
    def get_loc_type(loc):
        if loc.endswith('_S'): return 'Small'
        if loc.endswith('_M'): return 'Mid_Size'
        if loc.endswith('_L'): return 'Large'
        return 'Unknown'
    df['Location_Type'] = df['mapped_location'].apply(get_loc_type)

    df['Location_Type'] = pd.Categorical(
        df['Location_Type'],
        categories=['Mid_Size', 'Large', 'Small'],
        ordered=False
    )

    # Density (count per zipcode)
    df['zip_clinic_count'] = df.groupby('zip_clean')['zip_clean'].transform('count')

    # Distance to hub
    hub_coords = {
        'NYC_NY_L': (40.7128, -74.0060), 'Buffalo_NY_L': (42.8864, -78.8784), 'Syracuse_NY_M': (43.0481, -76.1474),
        'SaranacLake_NY_S': (44.3295, -74.1313), 'Malone_NY_S': (44.8487, -74.2963),
        'LA_CA_L': (34.0522, -118.2437), 'SanFrancisco_CA_L': (37.7749, -122.4194), 'Modesto_CA_M': (37.6391, -120.9969),
        'Eureka_CA_S': (40.8021, -124.1637), 'FortBragg_CA_S': (39.4457, -123.8053),
        'Atlanta_GA_L': (33.7490, -84.3880), 'Augusta_GA_L': (33.4735, -81.9665), 'Macon_GA_M': (32.8407, -83.6324),
        'Vidalia_GA_S': (32.2177, -82.4135), 'Toccoa_GA_S': (34.5771, -83.3324)
    }

    def get_distance(row):
        loc = row['mapped_location']
        if loc not in hub_coords: return np.nan
        h_lat, h_lon = hub_coords[loc]
        try:
            lon1, lat1, lon2, lat2 = map(radians, [float(row['longitude']), float(row['latitude']), h_lon, h_lat])
            dlon = lon2 - lon1; dlat = lat2 - lat1
            a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
            c = 2 * asin(sqrt(a))
            return c * 3956
        except: return np.nan

    if 'latitude' in df.columns:
        df['distance_miles'] = df.apply(get_distance, axis=1)

    # Preprocessing
    df['rating_value'] = pd.to_numeric(df['rating_value'], errors='coerce')
    df['log_votes'] = np.log1p(pd.to_numeric(df['votes_count'], errors='coerce').fillna(0))

    # Binary Outcome (Threshold <= 3.0)
    df['is_low_rating'] = np.where(df['rating_value'] <= 3.0, 1, 0)

    def get_state(loc):
        if '_NY_' in loc: return 'NY'
        if '_CA_' in loc: return 'CA'
        if '_GA_' in loc: return 'GA'
        return 'Other'
    df['State'] = df['mapped_location'].apply(get_state)

    reg_df = df.dropna(subset=['rating_value', 'distance_miles', 'log_votes', 'zip_clinic_count', 'State', 'Keyword_Category', 'Location_Type'])
    print(f"Data ready. N = {len(reg_df)}")

    # Regression
    common_formula = """
    distance_miles + log_votes + zip_clinic_count +
    State + Keyword_Category + Location_Type
    """

    # OLS
    print("\nOLS (Rating Value)")
    model_ols = smf.ols(formula=f"rating_value ~ {common_formula}", data=reg_df).fit()
    print(model_ols.summary())

    # Logit
    print("\nLogit (Low Rating Risk <= 3.0)")
    logit_res = smf.logit(formula=f"is_low_rating ~ {common_formula}", data=reg_df).fit()
    print(logit_res.summary())

    # Average Marginal Effects (AME)
    print("\n Average Marginal Effects (AME)")
    mfx = logit_res.get_margeff(at='overall', method='dydx')
    print(mfx.summary())

    # Accuracy
    preds = logit_res.predict()
    prediction_binary = (preds > 0.5).astype(int)
    actual = reg_df['is_low_rating']
    accuracy = (prediction_binary == actual).mean()
    print(f"\nPercent Correctly Predicted: {accuracy:.2%}")

# NPI records as Input

In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns

base_path = "/content/drive/MyDrive/RA/health_care/dentist_detailed_keywords"
nppes_data_path = "/content/drive/MyDrive/RA/health_care/NPI"
main_npi_file = "npidata_pfile_20050523-20251012.csv"

In [ ]:
# ZIP codes for each region
blacksburg_zips = [24060, 24061, 24062, 24063]
christiansburg_zips = [24068, 24073]
roanoke_lynchburg_zips = [
        # Roanoke ZIPs
        24001, 24002, 24003, 24004, 24005, 24006, 24007, 24008, 24009, 24010,
        24011, 24012, 24013, 24014, 24015, 24016, 24017, 24018, 24019, 24022,
        # Lynchburg ZIPs
        24501, 24502, 24503, 24504, 24505, 24506, 24513, 24514, 24515
    ]
dc_zips = [
        20001, 20002, 20003, 20004, 20005, 20006, 20007, 20008, 20009, 20010,
        20011, 20012, 20013, 20015, 20016, 20017, 20018, 20019, 20020, 20022,
        20023, 20024, 20026, 20027, 20029, 20030, 20032, 20033, 20035, 20036,
        20037, 20038, 20039, 20040, 20041, 20042, 20043, 20044, 20045, 20050,
        20051, 20052, 20053, 20055, 20056, 20057, 20058, 20059, 20060, 20061,
        20062, 20063, 20064, 20065, 20066, 20067, 20068, 20069, 20070, 20071,
        20073, 20074, 20075, 20076, 20077, 20078, 20080, 20081, 20082, 20090,
        20091, 20098,
        # NOVA ZIPs
        22301, 22302, 22303, 22304, 22305, 22306, 22307, 22308, 22309, 22310,
        22311, 22312, 22314, 22201, 22202, 22203, 22204, 22205, 22206, 22207,
        22209, 22211, 22213, 22214, 22041, 22042, 22043, 22044, 22046, 22003,
        22031, 22032, 22033, 22101, 22102, 22103, 22106, 22116, 22118, 22180,
        22181, 22182, 22067, 22039
    ]
ny_zips = [
        10001, 10002, 10003, 10004, 10005, 10006, 10007, 10008, 10009, 10010,
        10011, 10012, 10013, 10014, 10016, 10017, 10018, 10019, 10020, 10021,
        10022, 10023, 10024, 10025, 10026, 10027, 10028, 10029, 10030, 10031,
        10032, 10033, 10034, 10035, 10036, 10037, 10038, 10039, 10040, 10041,
        10043, 10044, 10045, 10046, 10047, 10048, 10055, 10060, 10065, 10069,
        10072, 10079, 10080, 10081, 10082, 10083, 10087, 10090, 10094, 10095,
        10096, 10098, 10099, 10101, 10102, 10103, 10104, 10105, 10106, 10107,
        10108, 10109, 10110, 10111, 10112, 10113, 10114, 10115, 10116, 10117,
        10118, 10119, 10120, 10121, 10122, 10123, 10124, 10125, 10126, 10128,
        10129, 10130, 10131, 10132, 10133, 10138, 10149, 10150, 10151, 10152,
        10153, 10154, 10155, 10156, 10157, 10158, 10159, 10160, 10161, 10162,
        10163, 10164, 10165, 10166, 10167, 10168, 10169, 10170, 10171, 10172,
        10173, 10174, 10175, 10176, 10177, 10178, 10179, 10184, 10185, 10196,
        10197, 10199, 10203, 10211, 10212, 10213, 10249, 10256, 10257, 10258,
        10259, 10260, 10261, 10265, 10268, 10269, 10270, 10271, 10272, 10273,
        10274, 10275, 10276, 10277, 10278, 10279, 10280, 10281, 10282, 10285,
        10286
    ]
charlotte_zips = [
        28201, 28202, 28203, 28204, 28205, 28206, 28207, 28208, 28209, 28210,
        28211, 28212, 28213, 28214, 28215, 28216, 28217, 28219, 28220, 28221,
        28222, 28223, 28224, 28226, 28227, 28228, 28229, 28230, 28231, 28232,
        28233, 28234, 28235, 28236, 28237, 28241, 28244, 28246, 28247, 28253,
        28254, 28255, 28256, 28258, 28260, 28262, 28265, 28266, 28269, 28270,
        28271, 28272, 28273, 28274, 28275, 28277, 28278, 28280, 28281, 28282,
        28284, 28285, 28287, 28296, 28299
]

all_target_zips = blacksburg_zips + christiansburg_zips + roanoke_lynchburg_zips + dc_zips + charlotte_zips + ny_zips

exclusion_taxonomy_codes = [
    # === Suppliers (DME, Pharmacy, Labs) ===
    '332B00000X',  # Durable Medical Equipment & Medical Supplies
    '332S00000X',  # Hearing Aid Equipment
    '333600000X',  # Pharmacy (General)
    '3336C0003X',  # Community/Retail Pharmacy
    '335E00000X',  # Prosthetic/Orthotic Supplier
    '335U00000X',  # Ocularist
    '291U00000X',  # Clinical Medical Laboratory

    # === Transportation Services ===
    '341600000X',  # Ambulance
    '343900000X',  # Non-emergency Medical Transport (VAN)
    '347B00000X',  # Taxi
    '347C00000X',  # Air Carrier

    # === Managed Care / Insurance ===
    '302F00000X',  # Health Maintenance Organization (HMO)
    '305S00000X',  # Preferred Provider Organization (PPO)

    # === Government Agencies ===
    '251K00000X',  # Public Health or Welfare Agency
    '251S00000X',  # Community Health
    '251T00000X',  # Military/U.S. Coast Guard Transport
    '251V00000X',  # Local Education Agency (LEA)

    # === Other Service & Social Work Providers ===
    '174400000X',  # Social Worker
    '1041C0700X',  # Clinical Social Worker
    '171M00000X',  # Case Manager/Care Coordinator

    # === Schools & Students ===
    '352Y00000X',  # Local Education Agency (as an organization)
    '390200000X',  # Student in an Organized Health Care Education/Training Program

    # === Residential & Long-Term Care Facilities ===
    '314000000X',  # Skilled Nursing Facility (SNF)
    '310400000X',  # Assisted Living Facility
    '320800000X',  # Residential Treatment Facility for Children
    '324500000X',  # Substance Abuse Rehabilitation Facility

    # === Community & Home Health Agencies ===
    '251E00000X',  # Home Health Agency
    '251C00000X',  # Community/Behavioral Health Agency
    '251J00000X',  # Voluntary or Charitable Agency
    '251F00000X'   # Hospice
]

# Taxonomy Code Distribution
print("--- Analyzing Taxonomy Code Distribution ---")
try:
    taxonomy_iterator = pd.read_csv(
        os.path.join(nppes_data_path, main_npi_file),
        usecols=['Healthcare Provider Taxonomy Code_1'],
        chunksize=100000,
        low_memory=False,
        encoding='utf-8'
    )
    taxonomy_counts = pd.Series(dtype='int64')
    for chunk in taxonomy_iterator:
        # Filter out excluded taxonomy codes before counting
        filtered_chunk = chunk[~chunk['Healthcare Provider Taxonomy Code_1'].isin(exclusion_taxonomy_codes)].copy()
        taxonomy_counts = taxonomy_counts.add(filtered_chunk['Healthcare Provider Taxonomy Code_1'].value_counts(), fill_value=0)

    taxonomy_counts = taxonomy_counts.sort_values(ascending=False)
    total_providers = taxonomy_counts.sum()
    taxonomy_percentage = (taxonomy_counts / total_providers) * 100

    print("\nTop 10 Healthcare Provider Taxonomy Categories:")
    top_10_taxonomy = taxonomy_percentage.head(10)
    print(top_10_taxonomy.to_string())

    plt.figure(figsize=(12, 8))
    sns.barplot(x=top_10_taxonomy.values, y=top_10_taxonomy.index, palette="viridis")
    plt.title('Top 10 Healthcare Provider Taxonomy Code Distribution', fontsize=16)
    plt.xlabel('Percentage (%)', fontsize=12)
    plt.ylabel('Taxonomy Code', fontsize=12)
    plt.tight_layout()
    plt.show()

except FileNotFoundError as e:
    print(f"\nError: Could not find the main NPI file for taxonomy analysis: {e}")

print("--- Processing Main NPI Data ---")
main_columns_to_load = [
    'NPI', 'Entity Type Code',
    'Provider Organization Name (Legal Business Name)',
    'Provider Last Name (Legal Name)', 'Provider First Name',
    'Provider Credential Text', 'Provider Enumeration Date',
    'Provider Business Practice Location Address Postal Code'
] + [f'Healthcare Provider Taxonomy Code_{i}' for i in range(1, 16)]

all_regional_data_chunks = []
print("Processing main NPI file in chunks to find all providers in target ZIPs...")

try:
    chunk_iterator = pd.read_csv(
        os.path.join(nppes_data_path, main_npi_file),
        usecols=main_columns_to_load,
        chunksize=100000,
        low_memory=False,
        encoding='utf-8'
    )

    for i, chunk in enumerate(chunk_iterator):
        print(f"  - Processing chunk {i+1}...")

        taxonomy_cols = [f'Healthcare Provider Taxonomy Code_{i}' for i in range(1, 16)]
        exclusion_mask = chunk[taxonomy_cols].isin(exclusion_taxonomy_codes).any(axis=1)
        filtered_chunk = chunk[~exclusion_mask].copy()

        if not filtered_chunk.empty:
            filtered_chunk['zip'] = filtered_chunk['Provider Business Practice Location Address Postal Code'].astype(str).str.split('-').str[0]
            filtered_chunk.dropna(subset=['zip'], inplace=True)
            filtered_chunk['zip'] = pd.to_numeric(filtered_chunk['zip'], errors='coerce')
            filtered_chunk.dropna(subset=['zip'], inplace=True)
            filtered_chunk['zip'] = filtered_chunk['zip'].astype(int)

            regional_providers = filtered_chunk[filtered_chunk['zip'].isin(all_target_zips)]

            if not regional_providers.empty:
                all_regional_data_chunks.append(regional_providers)

    if all_regional_data_chunks:
        df_regional = pd.concat(all_regional_data_chunks, ignore_index=True)


        df_regional['Name'] = df_regional['Provider Organization Name (Legal Business Name)'].copy()
        individual_mask = df_regional['Entity Type Code'] == 1
        df_regional.loc[individual_mask, 'Name'] = df_regional.loc[individual_mask, 'Provider First Name'] + ' ' + df_regional.loc[individual_mask, 'Provider Last Name (Legal Name)']

        # Rename columns
        df_regional.rename(columns={
            'Provider Enumeration Date': 'Registration_Date',
            'Provider Credential Text': 'Credential',
            'Healthcare Provider Taxonomy Code_1': 'Taxonomy_Code'
        }, inplace=True)

        def map_zip_to_location(zip_code):
            if zip_code in blacksburg_zips: return 'blacksburg'
            if zip_code in christiansburg_zips: return 'christiansburg'
            if zip_code in roanoke_lynchburg_zips: return 'roanoke_lynchburg'
            if zip_code in dc_zips: return 'washington_dc'
            if zip_code in charlotte_zips: return 'charlotte_nc'
            if zip_code in ny_zips: return 'new_york_ny'
            return 'Unknown'

        df_regional['location_name'] = df_regional['zip'].apply(map_zip_to_location)

        final_columns = [
            'NPI', 'Name', 'Registration_Date', 'Credential', 'zip', 'location_name', 'Taxonomy_Code'
        ]

        df_final_output = df_regional[final_columns].copy()
        df_final_output.dropna(subset=['Name'], inplace=True)
        df_final_output.drop_duplicates(subset=['NPI'], inplace=True)

        print(f"\nProcessing complete. Extracted {len(df_final_output)} unique providers from target regions.")

        print("\n--- Final Data Preview ---")
        display(df_final_output.head())

        # Optional: Save to a new CSV
        output_dir = os.path.join(base_path, "temp", "npi_processed")
        os.makedirs(output_dir, exist_ok=True)
        output_path = os.path.join(output_dir, "npi_processed.csv")
        df_final_output.to_csv(output_path, index=False, encoding='utf-8-sig')
        print(f"\nData saved to {output_path}")

    else:
        print("No regional providers found.")
        df_final_output = pd.DataFrame()

except FileNotFoundError as e:
    print(f"\nError: Could not find the main NPI file: {e}")
    df_final_output = pd.DataFrame()

In [ ]:
# dentist
import pandas as pd
from IPython.display import display

dentist_taxonomy_codes = [
    '122300000X',  # Dentist
    '1223D0001X',  # Dental Public Health
    '1223E0200X',  # Endodontics
    '1223G0001X',  # General Practice
    '1223P0106X',  # Pediatric Dentistry
    '1223P0221X',  # Periodontics
    '1223P0300X',  # Prosthodontics
    '1223P0700X',  # Pediatric Dentistry (alternate code)
    '1223S0112X',  # Oral and Maxillofacial Surgery
    '1223X0400X',  # Orthodontics and Dentofacial Orthopedics
    '122400000X',  # Dental Assistant
    '124Q00000X',  # Dental Hygienist
    '126800000X',  # Dental Laboratory
    '1223X2210X',  # Orofacial Pain
    '1223X0008X',  # Oral and Maxillofacial Radiology
    '1223P0106X',  # Oral and Maxillofacial Pathology
    '1223D0004X'   # Dentist Anesthesiologist
]

TAXONOMY_CATEGORIES = {
    'General': ['122300000X', '1223G0001X', '1223D0001X', '122400000X', '124Q00000X', '126800000X'],
    'Specialist': ['1223E0200X', '1223P0106X', '1223P0221X', '1223P0300X', '1223P0700X', '1223X0400X','1223X2210X','1223X0008X'],
    'Surgery': ['1223S0112X']
}

def get_npi_category(taxonomy_code):
    for category, codes in TAXONOMY_CATEGORIES.items():
        if taxonomy_code in codes:
            return category
    return 'Unknown'


if 'df_final_output' in locals() and not df_final_output.empty:
    print("Processing NPI filtered data...")
    df_dentists_filtered = df_final_output[df_final_output['Taxonomy_Code'].isin(dentist_taxonomy_codes)].copy()
    print(f"Filtered to {len(df_dentists_filtered)} dental-related providers.")
    df_dentists_filtered['category'] = df_dentists_filtered['Taxonomy_Code'].apply(get_npi_category)
    df_npi_keywords = df_dentists_filtered[[
        'Name', 'location_name', 'category', 'Registration_Date', 'Credential'
    ]].copy()

    df_npi_keywords.rename(columns={'Name': 'keyword'}, inplace=True)

    print(f"\nCreated 'df_npi_keywords' DataFrame with {len(df_npi_keywords)} entries for the next step.")
    print("\n--- Preview of the data to be posted ---")
    display(df_npi_keywords.head())

else:
    print("Error: 'df_final_output' not found or is empty. Please run the previous cell successfully.")
    df_npi_keywords = pd.DataFrame()

In [ ]:
# Post NPI Keywords to DataForSEO Maps API
print("--- Starting Task POST for NPI Keywords ---")

if 'df_npi_keywords' in locals() and not df_npi_keywords.empty:
    npi_search_temp_dir = os.path.join(base_path, "temp", "npi_map_search")
    os.makedirs(npi_search_temp_dir, exist_ok=True)
    task_list_csv_path = os.path.join(npi_search_temp_dir, "task_list.csv")

    # Load existing tasks to skip
    existing_tasks = set()
    try:
        if os.path.exists(task_list_csv_path):
            tasks_df = pd.read_csv(task_list_csv_path)
            if not tasks_df.empty:
                for index, row in tasks_df.iterrows():
                    existing_tasks.add((row['location_name'], row['keyword']))
            print(f"Loaded {len(existing_tasks)} existing tasks to skip.")
    except pd.errors.EmptyDataError:
        print("Task list file is empty. Starting fresh.")

    locations_to_search = {
        'roanoke_lynchburg': 200573,
        'blacksburg': 1027041,
        'christiansburg': 1027077,
        'washington_dc': 2840,
        'charlotte_nc': 200517,
        'new_york_ny' : 21167
    }

    maps_api_url = "https://api.dataforseo.com/v3/serp/google/maps/task_post"
    write_header = not os.path.exists(task_list_csv_path) or (os.path.getsize(task_list_csv_path) == 0)

    # Group keywords by their location
    grouped_keywords = df_npi_keywords.groupby('location_name')

    with open(task_list_csv_path, 'a', newline='', encoding='utf-8-sig') as f:
        import csv
        writer = csv.writer(f)
        if write_header:
            writer.writerow(["task_id", "api_type", "location_name", "category", "keyword",
                             "Registration_Date", "Credential", "raw_json_path"])
        # Iterate through each location group
        for loc_name, group in grouped_keywords:
            if loc_name not in locations_to_search:
                print(f"\nSkipping location '{loc_name}' as it has no corresponding location code.")
                continue

            loc_code = locations_to_search[loc_name]
            print(f"\nProcessing tasks for location: '{loc_name}' using code {loc_code}")

            for index, row in group.iterrows():
                keyword = row['keyword']
                category = row['category']
                reg_date = row['Registration_Date']
                credential = row['Credential']

                # Skip task if it already exists
                if (loc_name, keyword) in existing_tasks:
                    print(f"  Skipping already posted task for keyword: '{keyword}'")
                    continue

                task_id = post_dataforseo_task(maps_api_url, loc_code, keyword)

                if task_id:
                    safe_keyword = "".join(c for c in keyword if c.isalnum() or c in (' ', '_')).rstrip().replace(' ', '_')
                    raw_json_filename = f"{loc_name}_{category}_{safe_keyword[:100]}_maps.json"
                    raw_json_path = os.path.join(npi_search_temp_dir, raw_json_filename)

                    writer.writerow([task_id, "maps", loc_name, category, keyword,
                                     reg_date, credential, raw_json_path])
                    f.flush()

                time.sleep(1)

    print(f"\nAll targeted NPI keyword tasks posted.")
    print("!!! IMPORTANT: Please wait 20 minutes before running the next step. !!!")
else:
    print("Keyword DataFrame ('df_npi_keywords') is empty")

In [ ]:
base_path = "/content/drive/MyDrive/RA/health_care/dentist_detailed_keywords"
npi_search_temp_dir = os.path.join(base_path, "temp", "npi_map_search")
task_list_csv_path = os.path.join(npi_search_temp_dir, "task_list.csv")
final_output_dir = os.path.join(base_path, "output", "final")
final_csv_path = os.path.join(final_output_dir, "npi_map_search_final.csv")
os.makedirs(final_output_dir, exist_ok=True)

tasks_to_get = []
try:
    tasks_df = pd.read_csv(task_list_csv_path)
    tasks_to_get = tasks_df.to_dict('records')
    print(f"Found {len(tasks_to_get)} tasks in the task list.")

    for task in tasks_to_get:
        # Skip if the JSON file already exists
        if os.path.exists(task['raw_json_path']):
            print(f"  Skipping GET: Result file already exists for Keyword='{task['keyword']}'")
            continue

        print(f" Getting results for: Location='{task['location_name']}', Keyword='{task['keyword']}'")
        get_dataforseo_results(task['task_id'], task['api_type'], task['raw_json_path'])
        time.sleep(1)

    print("\nAll result retrieval attempts completed.")

except FileNotFoundError:
    print(f"Error: Task list file not found at '{task_list_csv_path}'. Cannot get results.")




In [ ]:
def parse_maps_results_with_category(file_path, category):
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        return []

    extracted_data = []
    if not (data and data.get("tasks") and data["tasks"][0].get("result")):
        return []

    for result in data["tasks"][0]["result"]:
        if not result or not result.get("items"):
            continue
        for item in result["items"]:
            rating = item.get("rating", {})
            if not isinstance(rating, dict): rating = {}
            address_info = item.get("address_info", {})
            if not isinstance(address_info, dict): address_info = {}
            rating_distribution = item.get("rating_distribution", {})
            if not isinstance(rating_distribution, dict): rating_distribution = {}

            record = {
                "title": item.get("title"),
                "address": item.get("address"),
                "latitude": item.get("latitude"),
                "longitude": item.get("longitude"),
                "zip": address_info.get("zip"),
                "rating_value": rating.get("value"),
                "votes_count": rating.get("votes_count"),
                "type": item.get("type"),
                "category": category  # Add the category to each record
            }
            # Add star ratings if they exist
            for i in range(1, 6):
                record[f'rating_{i}_star'] = rating_distribution.get(str(i), 0)

            extracted_data.append(record)
    return extracted_data



In [ ]:
# if not tasks_df.empty:
#     npi_maps_data = []
#     print(f"\nProcessing {len(tasks_df)} task results...")

#     for index, task_row in tasks_df.iterrows():
#         file_path = task_row['raw_json_path']
#         category = task_row['category']

#         if os.path.exists(file_path):
#             npi_maps_data.extend(parse_maps_results_with_category(file_path, category))
#         else:
#             print(f"  Warning: JSON file not found for task '{task_row['keyword']}', skipping.")

#     if npi_maps_data:
#         df_processed = pd.DataFrame(npi_maps_data)
#         print(f"Parsed {len(df_processed)} total records from all JSON files.")

#         df_deduplicated = df_processed.drop_duplicates(subset=['title', 'address'], keep='first').copy()
#         print(f"After deduplication, {len(df_deduplicated)} unique records remain.")

#         df_sorted = df_deduplicated.sort_values(by=['rating_value', 'votes_count'], ascending=[False, False], na_position='last')

#         final_columns = [
#             'title', 'category', 'address', 'latitude', 'longitude', 'zip',
#             'rating_value', 'votes_count', 'rating_1_star', 'rating_2_star',
#             'rating_3_star', 'rating_4_star', 'rating_5_star', 'type'
#         ]
#         for col in final_columns:
#             if col not in df_sorted.columns:
#                 df_sorted[col] = None

#         df_final_output = df_sorted[final_columns]

#         df_final_output.to_csv(final_csv_path, index=False, encoding='utf-8-sig')
#         print(f"\nProcessing complete. Final data saved to: {final_csv_path}")

#         print("\nPreview of the final processed NPI Map Search data:")
#         display(df_final_output.head())
#     else:
#         print("\nNo data was extracted from the JSON result files.")
# else:
#     print("\nNo tasks were found in the task list, so no data to process.")

In [ ]:
def load_processed_log(log_path):
    # Load the set of filenames that have already been processed
    try:
        with open(log_path, 'r') as f:
            return set(line.strip() for line in f)
    except FileNotFoundError:
        return set()

def update_processed_log(log_path, new_files):
    # Append newly processed filenames to the log
    with open(log_path, 'a') as f:
        for file_name in new_files:
            f.write(f"{file_name}\n")

def load_existing_data(csv_path):
    # Load the previously processed CSV data
    try:
        return pd.read_csv(csv_path)
    except (FileNotFoundError, pd.errors.EmptyDataError):
        return pd.DataFrame()

In [ ]:
if tasks_to_get:
    log_path = os.path.join(npi_search_temp_dir, "_processed_npi_files.log")

    # Load old data and log
    processed_files_set = load_processed_log(log_path)
    df_old = load_existing_data(final_csv_path)
    print(f"Loaded {len(processed_files_set)} processed file records from log.")
    print(f"Loaded {len(df_old)} existing records from '{os.path.basename(final_csv_path)}'.")

    # Process new files
    new_npi_maps_data = []
    files_processed_this_run = []

    print(f"\nProcessing {len(tasks_df)} task results...")
    for index, task_row in tasks_df.iterrows():
        file_path = task_row['raw_json_path']
        file_name = os.path.basename(file_path)
        category = task_row.get('category', 'Unknown')

        # Skip file already in log
        if file_name in processed_files_set:
            continue

        print(f"  Processing new file: {file_name}")

        if os.path.exists(file_path):
            parsed_data = parse_maps_results_with_category(file_path, category)
            if parsed_data:
                df_parsed = pd.DataFrame(parsed_data)

                df_parsed['Registration_Date'] = task_row.get('Registration_Date')
                df_parsed['Credential'] = task_row.get('Credential')

                new_npi_maps_data.append(df_parsed)
                files_processed_this_run.append(file_name)
        else:
            print(f"  Warning: JSON file not found for task '{task_row['keyword']}', skipping.")

    # Consolidate Data
    if new_npi_maps_data:
        df_new = pd.concat(new_npi_maps_data)
        print(f"\nParsed {len(df_new)} new records from {len(files_processed_this_run)} new files.")

        df_all = pd.concat([df_new, df_old], ignore_index=True)
        print(f"Total records before deduplication: {len(df_all)}")

        # Deduplicate
        df_deduplicated = df_all.drop_duplicates(subset=['title', 'address'], keep='first').copy()
        print(f"After deduplication, {len(df_deduplicated)} unique records remain.")

        df_sorted = df_deduplicated.sort_values(by=['rating_value', 'votes_count'], ascending=[False, False], na_position='last')

        final_columns = [
            'title', 'category', 'address', 'latitude', 'longitude', 'zip',
            'rating_value', 'votes_count', 'rating_1_star', 'rating_2_star',
            'rating_3_star', 'rating_4_star', 'rating_5_star', 'type','Registration_Date', 'Credential'
        ]
        for col in final_columns:
            if col not in df_sorted.columns:
                df_sorted[col] = None

        df_final_output = df_sorted[final_columns]

        # Overwrite the final CSV with the clean data
        df_final_output.to_csv(final_csv_path, index=False, encoding='utf-8-sig')
        print(f"\nProcessing complete. Final data saved to: {final_csv_path}")

        print("\nPreview of the final processed NPI Map Search data:")
        display(df_final_output.head())

        # Update the log file with the new files
        update_processed_log(log_path, files_processed_this_run)
        print(f"Updated '{os.path.basename(log_path)}' with {len(files_processed_this_run)} new files.")
    else:
        print("\nNo new files to process. Data in CSV remains unchanged.")
else:
    print("\nNo tasks were found in the task list, so no data to process.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np
from IPython.display import display

final_csv_path = "/content/drive/MyDrive/RA/health_care/dentist_detailed_keywords/output/final/npi_map_search_final.csv"

try:
    df = pd.read_csv(final_csv_path)
    print(f"Successfully loaded '{os.path.basename(final_csv_path)}'")
except FileNotFoundError:
    print(f"Error: The file '{final_csv_path}' was not found.")
    df = pd.DataFrame()

if not df.empty:
    blacksburg_zips = [24060, 24061, 24062, 24063]
    christiansburg_zips = [24068, 24073]
    roanoke_lynchburg_zips = [
        24001, 24002, 24003, 24004, 24005, 24006, 24007, 24008, 24009, 24010,
        24011, 24012, 24013, 24014, 24015, 24016, 24017, 24018, 24019, 24022,
        24501, 24502, 24503, 24504, 24505, 24506, 24513, 24514, 24515
    ]
    dc_zips = [
        20001, 20002, 20003, 20004, 20005, 20006, 20007, 20008, 20009, 20010,
        20011, 20012, 20013, 20015, 20016, 20017, 20018, 20019, 20020, 20022,
        20023, 20024, 20026, 20027, 20029, 20030, 20032, 20033, 20035, 20036,
        20037, 20038, 20039, 20040, 20041, 20042, 20043, 20044, 20045, 20050,
        20051, 20052, 20053, 20055, 20056, 20057, 20058, 20059, 20060, 20061,
        20062, 20063, 20064, 20065, 20066, 20067, 20068, 20069, 20070, 20071,
        20073, 20074, 20075, 20076, 20077, 20078, 20080, 20081, 20082, 20090,
        20091, 20098, 22301, 22302, 22303, 22304, 22305, 22306, 22307, 22308,
        22309, 22310, 22311, 22312, 22314, 22201, 22202, 22203, 22204, 22205,
        22206, 22207, 22209, 22211, 22213, 22214, 22041, 22042, 22043, 22044,
        22046, 22003, 22031, 22032, 22033, 22101, 22102, 22103, 22106, 22116,
        22118, 22180, 22181, 22182, 22067, 22039
    ]
    ny_zips = [
        10001, 10002, 10003, 10004, 10005, 10006, 10007, 10008, 10009, 10010,
        10011, 10012, 10013, 10014, 10016, 10017, 10018, 10019, 10020, 10021,
        10022, 10023, 10024, 10025, 10026, 10027, 10028, 10029, 10030, 10031,
        10032, 10033, 10034, 10035, 10036, 10037, 10038, 10039, 10040, 10041,
        10043, 10044, 10045, 10046, 10047, 10048, 10055, 10060, 10065, 10069,
        10072, 10079, 10080, 10081, 10082, 10083, 10087, 10090, 10094, 10095,
        10096, 10098, 10099, 10101, 10102, 10103, 10104, 10105, 10106, 10107,
        10108, 10109, 10110, 10111, 10112, 10113, 10114, 10115, 10116, 10117,
        10118, 10119, 10120, 10121, 10122, 10123, 10124, 10125, 10126, 10128,
        10129, 10130, 10131, 10132, 10133, 10138, 10149, 10150, 10151, 10152,
        10153, 10154, 10155, 10156, 10157, 10158, 10159, 10160, 10161, 10162,
        10163, 10164, 10165, 10166, 10167, 10168, 10169, 10170, 10171, 10172,
        10173, 10174, 10175, 10176, 10177, 10178, 10179, 10184, 10185, 10196,
        10197, 10199, 10203, 10211, 10212, 10213, 10249, 10256, 10257, 10258,
        10259, 10260, 10261, 10265, 10268, 10269, 10270, 10271, 10272, 10273,
        10274, 10275, 10276, 10277, 10278, 10279, 10280, 10281, 10282, 10285,
        10286
    ]
    charlotte_zips = [
        28201, 28202, 28203, 28204, 28205, 28206, 28207, 28208, 28209, 28210,
        28211, 28212, 28213, 28214, 28215, 28216, 28217, 28219, 28220, 28221,
        28222, 28223, 28224, 28226, 28227, 28228, 28229, 28230, 28231, 28232,
        28233, 28234, 28235, 28236, 28237, 28241, 28244, 28246, 28247, 28253,
        28254, 28255, 28256, 28258, 28260, 28262, 28265, 28266, 28269, 28270,
        28271, 28272, 28273, 28274, 28275, 28277, 28278, 28280, 28281, 28282,
        28284, 28285, 28287, 28296, 28299
    ]

    def map_zip_to_location(zip_code):
        if pd.isna(zip_code): return 'Unknown'
        try:
            zip_code = int(zip_code)
        except ValueError:
            return 'Unknown'
        if zip_code in blacksburg_zips: return 'Blacksburg'
        if zip_code in christiansburg_zips: return 'Christiansburg'
        if zip_code in roanoke_lynchburg_zips: return 'Roanoke/Lynchburg'
        if zip_code in dc_zips: return 'DC'
        if zip_code in ny_zips: return 'NY'
        if zip_code in charlotte_zips: return 'Charlotte'
        return 'Unknown'

    df['location_name'] = df['zip'].apply(map_zip_to_location)

    df_filtered = df[(df['category'] != 'Unknown') & (df['location_name'] != 'Unknown')].copy()
    df_filtered['rating_value'] = pd.to_numeric(df_filtered['rating_value'], errors='coerce')
    df_filtered.dropna(subset=['rating_value', 'category', 'location_name'], inplace=True)
    df_filtered['low_rating_dummy'] = np.where(df_filtered['rating_value'] <= 3, 1, 0)
    urban_locations = ['DC', 'NY', 'Charlotte']
    df_filtered['location_type'] = df_filtered['location_name'].apply(lambda x: 'Urban' if x in urban_locations else 'Rural') # Corrected 'Country' to 'Rural' for consistency
    star_cols = ['rating_1_star', 'rating_2_star', 'rating_3_star', 'rating_4_star', 'rating_5_star']
    for col in star_cols:
        df_filtered[col] = pd.to_numeric(df_filtered[col], errors='coerce').fillna(0)
    df_filtered['rating_std'] = df_filtered[star_cols].std(axis=1)

    print(f"After all filtering and preparation, {len(df_filtered)} records remain for analysis.")


    # Boxplot
    categories = sorted(df_filtered['category'].unique())
    locations_order = sorted(df_filtered['location_name'].unique())

    print(f"---Generating boxplots for {len(categories)} categories---")

    for category in categories:
        plt.figure(figsize=(12, 7))
        category_df = df_filtered[df_filtered['category'] == category]

        sns.boxplot(x='location_name', y='rating_value', data=category_df, order=locations_order)

        plt.title(f'Rating Distribution for: {category}', fontsize=16)
        plt.xlabel('Region', fontsize=12)
        plt.ylabel('Rating Value', fontsize=12)
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

    # Histograms each region
    print(f"---Generating Histograms for {len(categories)} categories---")
    for category in categories:
        for location in locations_order:
            plt.figure(figsize=(10, 6))

            plot_df = df_filtered[
                (df_filtered['category'] == category) &
                (df_filtered['location_name'] == location)
            ]

            if plot_df.empty:
                print(f"  Skipping histogram for '{category}' in '{location}' (No data)")
                plt.close()
                continue

            sns.histplot(data=plot_df, x='rating_value', bins=15, kde=True, color='blue')

            plt.title(f'Histogram of Ratings for: {category} in {location}')
            plt.xlabel('Rating Value')
            plt.ylabel('Count of Clinics')
            plt.xlim(1, 5)
            plt.tight_layout()
            plt.show()

    # Violin Plot of Rating Value
    print(f"---Generating Violin Plot for {len(categories)} categories---")
    for category in categories:
        plt.figure(figsize=(12, 7))
        category_df = df_filtered[df_filtered['category'] == category]
        sns.violinplot(x='location_name', y='rating_value', data=category_df, order=locations_order)
        plt.title(f'Rating Distribution for: {category}', fontsize=16)
        plt.xlabel('Region', fontsize=12)
        plt.ylabel('Rating Value', fontsize=12)
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

    # low rate crosstab
    low_rating_crosstab = pd.crosstab(df_filtered['location_type'], df_filtered['low_rating_dummy'])
    print("--- Low Rating (<=3) Analysis: Urban vs. Rural ---")
    display(low_rating_crosstab)

    low_rating_crosstab.plot(kind='bar', stacked=True, figsize=(8, 6), color=['skyblue', 'salmon'])
    plt.title('Count of Low Ratings (<=3) by Location Type')
    plt.xlabel('Location Type')
    plt.ylabel('Number of Clinics')
    plt.xticks(rotation=0)
    plt.legend(title='Low Rating (<=3)', labels=['No (>3)', 'Yes (<=3)'])
    plt.tight_layout()
    plt.show()

    #std
    std_by_category_location_series = df_filtered.groupby(['category', 'location_name'])['rating_std'].mean()
    print(std_by_category_location_series.unstack())
    std_by_category_location = df_filtered.groupby(['category', 'location_name'], as_index=False)['rating_std'].mean()
    print("--- Average Rating Standard Deviation by Category and Location ---")


    if not std_by_category_location.empty:
        for category in categories:
            plt.figure(figsize=(12, 7))

            plot_df = std_by_category_location[std_by_category_location['category'] == category]

            if plot_df.empty:
                print(f"  Skipping std dev plot for '{category}' (No data)")
                plt.close()
                continue

            # Applied fix
            sns.barplot(x='location_name', y='rating_std', data=plot_df, order=locations_order, palette="coolwarm", hue='location_name', legend=False)

            plt.title(f'Average Rating Standard Deviation for: {category}', fontsize=16)
            plt.xlabel('Region', fontsize=12)
            plt.ylabel('Average Standard Deviation of Star Counts', fontsize=12)
            plt.xticks(rotation=45, ha='right')
            plt.tight_layout()
            plt.savefig(f'std_dev_ratings_cat_{category}.png')
            plt.show()
    else:
        print("Could not generate standard deviation data from groupby.")

#Merge

In [ ]:
import pandas as pd
import numpy as np
import os
from IPython.display import display

base_path = "/content/drive/MyDrive/RA/health_care/dentist_detailed_keywords"
google_data_path = os.path.join(base_path, "output", "final", "final_processed.csv")
npi_data_path = os.path.join(base_path, "output", "final", "npi_map_search_final.csv")

output_dir = os.path.join(base_path, "output", "final_merged")
os.makedirs(output_dir, exist_ok=True)
intersection_output_path = os.path.join(output_dir, "merged_intersection_final.csv")
union_output_path = os.path.join(output_dir, "merged_union_final.csv")

try:
    df_google = pd.read_csv(google_data_path)
    df_npi = pd.read_csv(npi_data_path)

    print(f"Successfully loaded 'final_processed.csv' ({len(df_google)} records)")
    print(f"Successfully loaded 'npi_map_search_final.csv' ({len(df_npi)} records)")

except FileNotFoundError as e:
    print(f"Error loading files: {e}")
    df_google = pd.DataFrame()
    df_npi = pd.DataFrame()

if not df_google.empty and not df_npi.empty:
    category_map = {
        'keywords_General_Dentist': 'General',
        'keywords_Special_Dentist': 'Specialist',
        'keywords_Surgery_Dentist': 'Surgery'
    }

    df_google['category'] = df_google['category'].map(category_map).fillna(df_google['category'])
    print("\nStandardized 'category' column in Google data.")

    FINAL_COLUMNS = [
        'title', 'category', 'address', 'latitude', 'longitude', 'zip',
        'rating_value', 'votes_count',
        'rating_1_star', 'rating_2_star', 'rating_3_star', 'rating_4_star', 'rating_5_star',
        'isFinder', 'isMap', 'isNPI', 'isAll',
        'Registration_Date', 'Credential'
    ]

    # Outer Join
    print("--- Union ---")
    if 'isBoth' in df_google.columns:
        df_google.rename(columns={'isBoth': 'isGoogleBoth'}, inplace=True)

    df_union = pd.merge(
        df_google,
        df_npi,
        on='title',
        how='outer',
        suffixes=('_google', '_npi')
    )

    shared_cols = [ 'address', 'latitude', 'longitude', 'zip', 'rating_value', 'votes_count',
                   'rating_1_star', 'rating_2_star', 'rating_3_star', 'rating_4_star', 'rating_5_star']

    for col in shared_cols:
        df_union[col] = df_union[f'{col}_google'].fillna(df_union[f'{col}_npi'])

    df_union['category'] = df_union['category_google']
    df_union['category'] = df_union['category'].mask(
        (df_union['category'].isna()) | (df_union['category'] == 'Unknown'),
        df_union['category_npi']
    )


    is_from_google = df_union['isFinder'].notna()
    is_from_npi = df_union['Registration_Date'].notna()

    df_union['isFinder'] = df_union['isFinder'].fillna(0).astype(int)

    is_map_google = df_union['isMap'].fillna(0).astype(int)
    df_union['isMap'] = (is_map_google | is_from_npi).astype(int)

    df_union['isNPI'] = is_from_npi.astype(int)

    df_union['isAll'] = (is_from_google & is_from_npi).astype(int)

    for col in FINAL_COLUMNS:
        if col not in df_union.columns:
            df_union[col] = np.nan

    # deduplication
    df_union_final = df_union[FINAL_COLUMNS].copy()

    print(f"Union before deduplication: {len(df_union_final)} records")
    df_union_final = df_union_final.sort_values(by=['isAll', 'isFinder'], ascending=[False, False])
    df_union_final = df_union_final.drop_duplicates(subset=['title', 'address'], keep='first')
    print(f"Union after deduplication: {len(df_union_final)} records")


    print(f"Union resulted in {len(df_union_final)} total unique records.")
    df_union_final.to_csv(union_output_path, index=False, encoding='utf-8-sig')
    print(f"Saved union file to: {union_output_path}")

    # Intersection
    print("---Intersection ---")

    df_intersection_final = df_union_final[df_union_final['isAll'] == 1].copy()

    print(f"Intersection resulted in {len(df_intersection_final)} records.")
    df_intersection_final.to_csv(intersection_output_path, index=False, encoding='utf-8-sig')
    print(f"Saved intersection file to: {intersection_output_path}")

    print("\n--- Merge Complete ---")
    print("\nIntersection Preview:")
    display(df_intersection_final.head())
    print("\nUnion Preview:")
    display(df_union_final.head())

else:
    print("\nOne or more DataFrames are empty. Merge operation skipped.")

# Check vote counts vs. ratings

In [ ]:
import pandas as pd
import numpy as np
import os
from IPython.display import display

base_path = "/content/drive/MyDrive/RA/health_care/dentist_detailed_keywords"
output_dir = os.path.join(base_path, "output", "final_merged")
union_input_path = os.path.join(output_dir, "merged_union_final.csv")
discrepancy_output_path = os.path.join(output_dir, "discrepancy_analysis.csv")

try:
    df = pd.read_csv(union_input_path)
    print(f"Successfully loaded 'merged_union_final.csv' ({len(df)} records)")
except FileNotFoundError as e:
    print(f"Error loading files: {e}")
    df = pd.DataFrame()

if not df.empty:
    star_cols = ['rating_1_star', 'rating_2_star', 'rating_3_star', 'rating_4_star', 'rating_5_star']
    cols_to_clean = ['votes_count'] + star_cols

    for col in cols_to_clean:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

    print("Cleaned all rating and vote count columns.")


    df['star_sum'] = df[star_cols].sum(axis=1)
    df['difference'] = df['votes_count'] - df['star_sum']

    df['difference_pct'] = np.where(
        df['votes_count'] > 0,
        (df['difference'] / df['votes_count']) * 100,
        0
    )

    # 0 = normal, 1 = vote>0 rate=0, 2 = rate>0 vote!=rate
    df['discrepancy_type'] = 0

    # 1
    df.loc[
        (df['votes_count'] > 0) & (df['star_sum'] == 0),
        'discrepancy_type'
    ] = 1

    # 2
    df.loc[
        (df['star_sum'] > 0) & (df['votes_count'] != df['star_sum']),
        'discrepancy_type'
    ] = 2

    print("\n--- Discrepancy Type Counts ---")
    print(df['discrepancy_type'].value_counts())


    df_discrepancies = df[df['discrepancy_type'] > 0].copy()

    if not df_discrepancies.empty:
        print(f"\nFound {len(df_discrepancies)} records with discrepancies.")
        df_discrepancies.to_csv(discrepancy_output_path, index=False, encoding='utf-8-sig')
        print(f"Saved all discrepancy records to: {discrepancy_output_path}")

        print("--- Type 1 ---")
        display(df_discrepancies[df_discrepancies['discrepancy_type'] == 1][['votes_count', 'difference', 'difference_pct']].describe())

        print("--- Type 2 ---")
        display(df_discrepancies[df_discrepancies['discrepancy_type'] == 2][['votes_count', 'star_sum', 'difference', 'difference_pct']].describe())


        cols_to_show = ['title', 'votes_count', 'star_sum', 'difference', 'difference_pct', 'discrepancy_type', 'category', 'address']

        print("--- Type 1 ---")
        display(df_discrepancies[df_discrepancies['discrepancy_type'] == 1][cols_to_show].head(20))

        print("--- Type 2 ---")
        display(df_discrepancies[df_discrepancies['discrepancy_type'] == 2][cols_to_show].head(20))

    else:
        print("\nNo discrepancies found in the data.")

else:
    print("\nDataFrame is empty. Analysis skipped.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np

final_csv_path = "/content/drive/MyDrive/RA/health_care/dentist_detailed_keywords/output/final_merged/merged_union_final.csv"


try:
    df = pd.read_csv(final_csv_path)
    print("Successfully loaded final_processed.csv")
except FileNotFoundError:
    print(f"Error: The file '{final_csv_path}' was not found.")
    df = pd.DataFrame()

if not df.empty:
    # ZIP codes for each region
    blacksburg_zips = [24060, 24061, 24062, 24063]
    christiansburg_zips = [24068, 24073]
    roanoke_lynchburg_zips = [
        # Roanoke ZIPs
        24001, 24002, 24003, 24004, 24005, 24006, 24007, 24008, 24009, 24010,
        24011, 24012, 24013, 24014, 24015, 24016, 24017, 24018, 24019, 24022,
        # Lynchburg ZIPs
        24501, 24502, 24503, 24504, 24505, 24506, 24513, 24514, 24515
    ]
    dc_zips = [
        20001, 20002, 20003, 20004, 20005, 20006, 20007, 20008, 20009, 20010,
        20011, 20012, 20013, 20015, 20016, 20017, 20018, 20019, 20020, 20022,
        20023, 20024, 20026, 20027, 20029, 20030, 20032, 20033, 20035, 20036,
        20037, 20038, 20039, 20040, 20041, 20042, 20043, 20044, 20045, 20050,
        20051, 20052, 20053, 20055, 20056, 20057, 20058, 20059, 20060, 20061,
        20062, 20063, 20064, 20065, 20066, 20067, 20068, 20069, 20070, 20071,
        20073, 20074, 20075, 20076, 20077, 20078, 20080, 20081, 20082, 20090,
        20091, 20098,
        # NOVA) ZIP
        22301, 22302, 22303, 22304, 22305, 22306, 22307, 22308, 22309, 22310,
        22311, 22312, 22314, 22201, 22202, 22203, 22204, 22205, 22206, 22207,
        22209, 22211, 22213, 22214, 22041, 22042, 22043, 22044, 22046, 22003,
        22031, 22032, 22033, 22101, 22102, 22103, 22106, 22116, 22118, 22180,
        22181, 22182, 22067, 22039
    ]
    ny_zips = [
        10001, 10002, 10003, 10004, 10005, 10006, 10007, 10008, 10009, 10010,
        10011, 10012, 10013, 10014, 10016, 10017, 10018, 10019, 10020, 10021,
        10022, 10023, 10024, 10025, 10026, 10027, 10028, 10029, 10030, 10031,
        10032, 10033, 10034, 10035, 10036, 10037, 10038, 10039, 10040, 10041,
        10043, 10044, 10045, 10046, 10047, 10048, 10055, 10060, 10065, 10069,
        10072, 10079, 10080, 10081, 10082, 10083, 10087, 10090, 10094, 10095,
        10096, 10098, 10099, 10101, 10102, 10103, 10104, 10105, 10106, 10107,
        10108, 10109, 10110, 10111, 10112, 10113, 10114, 10115, 10116, 10117,
        10118, 10119, 10120, 10121, 10122, 10123, 10124, 10125, 10126, 10128,
        10129, 10130, 10131, 10132, 10133, 10138, 10149, 10150, 10151, 10152,
        10153, 10154, 10155, 10156, 10157, 10158, 10159, 10160, 10161, 10162,
        10163, 10164, 10165, 10166, 10167, 10168, 10169, 10170, 10171, 10172,
        10173, 10174, 10175, 10176, 10177, 10178, 10179, 10184, 10185, 10196,
        10197, 10199, 10203, 10211, 10212, 10213, 10249, 10256, 10257, 10258,
        10259, 10260, 10261, 10265, 10268, 10269, 10270, 10271, 10272, 10273,
        10274, 10275, 10276, 10277, 10278, 10279, 10280, 10281, 10282, 10285,
        10286
    ]
    charlotte_zips = [
        28201, 28202, 28203, 28204, 28205, 28206, 28207, 28208, 28209, 28210,
        28211, 28212, 28213, 28214, 28215, 28216, 28217, 28219, 28220, 28221,
        28222, 28223, 28224, 28226, 28227, 28228, 28229, 28230, 28231, 28232,
        28233, 28234, 28235, 28236, 28237, 28241, 28244, 28246, 28247, 28253,
        28254, 28255, 28256, 28258, 28260, 28262, 28265, 28266, 28269, 28270,
        28271, 28272, 28273, 28274, 28275, 28277, 28278, 28280, 28281, 28282,
        28284, 28285, 28287, 28296, 28299
    ]

    # map ZIP code to location
    def map_zip_to_location(zip_code):
        if pd.isna(zip_code): return 'Unknown'
        try:
            zip_code = int(zip_code)
        except ValueError:
            return 'Unknown'

        if zip_code in blacksburg_zips:
            return 'Blacksburg'
        elif zip_code in christiansburg_zips:
            return 'Christiansburg'
        elif zip_code in roanoke_lynchburg_zips:
            return 'Roanoke/Lynchburg'
        elif zip_code in dc_zips:
            return 'DC'
        elif zip_code in ny_zips:
            return 'NY'
        elif zip_code in charlotte_zips:
            return 'Charlotte'
        return 'Unknown'

    df['location_name'] = df['zip'].apply(map_zip_to_location)

    df_filtered = df[(df['category'] != 'Unknown') & (df['location_name'] != 'Unknown')].copy()
    df_filtered['rating_value'] = pd.to_numeric(df_filtered['rating_value'], errors='coerce')
    df_filtered.dropna(subset=['rating_value', 'category', 'location_name'], inplace=True)
    # low rate dummy
    df_filtered['low_rating_dummy'] = np.where(df_filtered['rating_value'] <= 3, 1, 0)
    urban_locations = ['DC', 'NY', 'Charlotte']
    df_filtered['location_type'] = df_filtered['location_name'].apply(lambda x: 'Urban' if x in urban_locations else 'Country')
    # std rate 1-5
    star_cols = ['rating_1_star', 'rating_2_star', 'rating_3_star', 'rating_4_star', 'rating_5_star']
    for col in star_cols:
        df_filtered[col] = pd.to_numeric(df_filtered[col], errors='coerce').fillna(0)
    df_filtered['rating_std'] = df_filtered[star_cols].std(axis=1)

    print(f"After all filtering and preparation, {len(df_filtered)} records remain for analysis.")


    # Boxplot
    categories = sorted(df_filtered['category'].unique())
    locations_order = sorted(df_filtered['location_name'].unique())

    print(f"---Generating boxplots for {len(categories)} categories---")

    for category in categories:
        plt.figure(figsize=(12, 7))
        category_df = df_filtered[df_filtered['category'] == category]

        sns.boxplot(x='location_name', y='rating_value', data=category_df, order=locations_order)

        plt.title(f'Rating Distribution for: {category}', fontsize=16)
        plt.xlabel('Region', fontsize=12)
        plt.ylabel('Rating Value', fontsize=12)
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

    # Histograms each region
    print(f"---Generating Histograms for {len(categories)} categories---")
    for category in categories:
        for location in locations_order:
            plt.figure(figsize=(10, 6))

            plot_df = df_filtered[
                (df_filtered['category'] == category) &
                (df_filtered['location_name'] == location)
            ]

            if plot_df.empty:
                print(f"  Skipping histogram for '{category}' in '{location}' (No data)")
                plt.close()
                continue

            sns.histplot(data=plot_df, x='rating_value', bins=15, kde=True, color='blue')

            plt.title(f'Histogram of Ratings for: {category} in {location}')
            plt.xlabel('Rating Value')
            plt.ylabel('Count of Clinics')
            plt.xlim(1, 5)
            plt.tight_layout()
            plt.show()

    # Violin Plot of Rating Value
    print(f"---Generating Violin Plot for {len(categories)} categories---")
    for category in categories:
        plt.figure(figsize=(12, 7))
        category_df = df_filtered[df_filtered['category'] == category]
        sns.violinplot(x='location_name', y='rating_value', data=category_df, order=locations_order)
        plt.title(f'Rating Distribution for: {category}', fontsize=16)
        plt.xlabel('Region', fontsize=12)
        plt.ylabel('Rating Value', fontsize=12)
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

    # low rate crosstab
    low_rating_crosstab = pd.crosstab(df_filtered['location_type'], df_filtered['low_rating_dummy'])
    print("--- Low Rating (<=3) Analysis: Urban vs. Rural ---")
    display(low_rating_crosstab)

    low_rating_crosstab.plot(kind='bar', stacked=True, figsize=(8, 6), color=['skyblue', 'salmon'])
    plt.title('Count of Low Ratings (<=3) by Location Type')
    plt.xlabel('Location Type')
    plt.ylabel('Number of Clinics')
    plt.xticks(rotation=0)
    plt.legend(title='Low Rating (<=3)', labels=['No (>3)', 'Yes (<=3)'])
    plt.tight_layout()
    plt.show()

    #std
    std_by_category_location_series = df_filtered.groupby(['category', 'location_name'])['rating_std'].mean()
    print(std_by_category_location_series.unstack())
    std_by_category_location = df_filtered.groupby(['category', 'location_name'], as_index=False)['rating_std'].mean()
    print("--- Average Rating Standard Deviation by Category and Location ---")


    if not std_by_category_location.empty:
        for category in categories:
            plt.figure(figsize=(12, 7))

            plot_df = std_by_category_location[std_by_category_location['category'] == category]

            if plot_df.empty:
                print(f"  Skipping std dev plot for '{category}' (No data)")
                plt.close()
                continue

            # Applied fix
            sns.barplot(x='location_name', y='rating_std', data=plot_df, order=locations_order, palette="coolwarm", hue='location_name', legend=False)

            plt.title(f'Average Rating Standard Deviation for: {category}', fontsize=16)
            plt.xlabel('Region', fontsize=12)
            plt.ylabel('Average Standard Deviation of Star Counts', fontsize=12)
            plt.xticks(rotation=45, ha='right')
            plt.tight_layout()
            plt.savefig(f'std_dev_ratings_cat_{category}.png')
            plt.show()
    else:
        print("Could not generate standard deviation data from groupby.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np
import math

final_csv_path = "/content/drive/MyDrive/RA/health_care/dentist_detailed_keywords/output/final_merged/merged_union_final.csv"
fig_counter = 1

def get_fig_title(title_text):
    global fig_counter
    title = f"Figure {fig_counter}: {title_text}"
    fig_counter += 1
    return title

try:
    df = pd.read_csv(final_csv_path)
    print("Successfully loaded final_processed.csv")
except FileNotFoundError:
    print(f"Error: The file '{final_csv_path}' was not found.")
    df = pd.DataFrame()

if not df.empty:
    blacksburg_zips = [24060, 24061, 24062, 24063]
    christiansburg_zips = [24068, 24073]
    roanoke_lynchburg_zips = [
        # Roanoke ZIPs
        24001, 24002, 24003, 24004, 24005, 24006, 24007, 24008, 24009, 24010,
        24011, 24012, 24013, 24014, 24015, 24016, 24017, 24018, 24019, 24022,
        # Lynchburg ZIPs
        24501, 24502, 24503, 24504, 24505, 24506, 24513, 24514, 24515
    ]
    dc_zips = [
        20001, 20002, 20003, 20004, 20005, 20006, 20007, 20008, 20009, 20010,
        20011, 20012, 20013, 20015, 20016, 20017, 20018, 20019, 20020, 20022,
        20023, 20024, 20026, 20027, 20029, 20030, 20032, 20033, 20035, 20036,
        20037, 20038, 20039, 20040, 20041, 20042, 20043, 20044, 20045, 20050,
        20051, 20052, 20053, 20055, 20056, 20057, 20058, 20059, 20060, 20061,
        20062, 20063, 20064, 20065, 20066, 20067, 20068, 20069, 20070, 20071,
        20073, 20074, 20075, 20076, 20077, 20078, 20080, 20081, 20082, 20090,
        20091, 20098,
        # NOVA) ZIP
        22301, 22302, 22303, 22304, 22305, 22306, 22307, 22308, 22309, 22310,
        22311, 22312, 22314, 22201, 22202, 22203, 22204, 22205, 22206, 22207,
        22209, 22211, 22213, 22214, 22041, 22042, 22043, 22044, 22046, 22003,
        22031, 22032, 22033, 22101, 22102, 22103, 22106, 22116, 22118, 22180,
        22181, 22182, 22067, 22039
    ]
    ny_zips = [
        10001, 10002, 10003, 10004, 10005, 10006, 10007, 10008, 10009, 10010,
        10011, 10012, 10013, 10014, 10016, 10017, 10018, 10019, 10020, 10021,
        10022, 10023, 10024, 10025, 10026, 10027, 10028, 10029, 10030, 10031,
        10032, 10033, 10034, 10035, 10036, 10037, 10038, 10039, 10040, 10041,
        10043, 10044, 10045, 10046, 10047, 10048, 10055, 10060, 10065, 10069,
        10072, 10079, 10080, 10081, 10082, 10083, 10087, 10090, 10094, 10095,
        10096, 10098, 10099, 10101, 10102, 10103, 10104, 10105, 10106, 10107,
        10108, 10109, 10110, 10111, 10112, 10113, 10114, 10115, 10116, 10117,
        10118, 10119, 10120, 10121, 10122, 10123, 10124, 10125, 10126, 10128,
        10129, 10130, 10131, 10132, 10133, 10138, 10149, 10150, 10151, 10152,
        10153, 10154, 10155, 10156, 10157, 10158, 10159, 10160, 10161, 10162,
        10163, 10164, 10165, 10166, 10167, 10168, 10169, 10170, 10171, 10172,
        10173, 10174, 10175, 10176, 10177, 10178, 10179, 10184, 10185, 10196,
        10197, 10199, 10203, 10211, 10212, 10213, 10249, 10256, 10257, 10258,
        10259, 10260, 10261, 10265, 10268, 10269, 10270, 10271, 10272, 10273,
        10274, 10275, 10276, 10277, 10278, 10279, 10280, 10281, 10282, 10285,
        10286
    ]
    charlotte_zips = [
        28201, 28202, 28203, 28204, 28205, 28206, 28207, 28208, 28209, 28210,
        28211, 28212, 28213, 28214, 28215, 28216, 28217, 28219, 28220, 28221,
        28222, 28223, 28224, 28226, 28227, 28228, 28229, 28230, 28231, 28232,
        28233, 28234, 28235, 28236, 28237, 28241, 28244, 28246, 28247, 28253,
        28254, 28255, 28256, 28258, 28260, 28262, 28265, 28266, 28269, 28270,
        28271, 28272, 28273, 28274, 28275, 28277, 28278, 28280, 28281, 28282,
        28284, 28285, 28287, 28296, 28299
    ]

    def map_zip_to_location(zip_code):
        if pd.isna(zip_code): return 'Unknown'
        try:
            zip_code = int(zip_code)
        except ValueError:
            return 'Unknown'

        if zip_code in blacksburg_zips: return 'Blacksburg'
        elif zip_code in christiansburg_zips: return 'Christiansburg'
        elif zip_code in roanoke_lynchburg_zips: return 'Roanoke/Lynchburg'
        elif zip_code in dc_zips: return 'DC'
        elif zip_code in ny_zips: return 'NY'
        elif zip_code in charlotte_zips: return 'Charlotte'
        return 'Unknown'

    df['location_name'] = df['zip'].apply(map_zip_to_location)
    df_filtered = df[(df['category'] != 'Unknown') & (df['location_name'] != 'Unknown')].copy()

    # Clean star columns
    star_cols = ['rating_1_star', 'rating_2_star', 'rating_3_star', 'rating_4_star', 'rating_5_star']
    for col in star_cols:
        df_filtered[col] = pd.to_numeric(df_filtered[col], errors='coerce').fillna(0).astype(int)

    # !!!!!!!!!deleted all data points that "could not generate image", which means the image doesn't cover all data points
    df_filtered['rating_value'] = pd.to_numeric(df_filtered['rating_value'], errors='coerce')
    df_filtered.dropna(subset=['rating_value', 'category', 'location_name'], inplace=True)

    urban_locations = ['DC', 'NY', 'Charlotte']
    df_filtered['location_type'] = df_filtered['location_name'].apply(lambda x: 'Urban' if x in urban_locations else 'Country')

    # Calculate std, P25, P75, 1.5IQR
    def calculate_metrics(row):
        ratings_distribution = []
        ratings_distribution.extend([1] * row['rating_1_star'])
        ratings_distribution.extend([2] * row['rating_2_star'])
        ratings_distribution.extend([3] * row['rating_3_star'])
        ratings_distribution.extend([4] * row['rating_4_star'])
        ratings_distribution.extend([5] * row['rating_5_star'])

        if not ratings_distribution:
            return pd.Series([np.nan, np.nan, np.nan, np.nan])

        std_dev = np.std(ratings_distribution, ddof=1) if len(ratings_distribution) > 1 else 0
        p10 = np.percentile(ratings_distribution, 10)
        p25 = np.percentile(ratings_distribution, 25)
        p75 = np.percentile(ratings_distribution, 75)
        iqr = p75 - p25

        return pd.Series([std_dev, p10, p25, iqr])

    metrics_df = df_filtered.apply(calculate_metrics, axis=1)
    metrics_df.columns = ['std', 'p10', 'p25', 'iqr']

    df_filtered = pd.concat([df_filtered, metrics_df], axis=1)

    # Low-Rating Thresholds
    df_filtered['low_rating_by_avg'] = np.where(df_filtered['rating_value'] <= 4, 1, 0)
    df_filtered['low_rating_by_p25'] = np.where(df_filtered['p25'] <= 4, 1, 0)

    print(f"Data ready. {len(df_filtered)} records.")

    categories = sorted(df_filtered['category'].unique())
    locations_order = sorted(df_filtered['location_name'].unique())



    for category in categories:
        print(f"Processing Category: {category}")
        category_df = df_filtered[df_filtered['category'] == category]

        # Boxplot (Avg, Std)
        fig, axes = plt.subplots(1, 2, figsize=(16, 7))

        # Avg Boxplot
        sns.boxplot(x='location_name', y='rating_value', data=category_df, order=locations_order, ax=axes[0], palette='Set1', hue='location_name', legend=False)
        axes[0].set_title(f'Average Rating', fontsize=12)
        axes[0].set_xlabel('Region')
        axes[0].set_ylabel('Rating Value')
        axes[0].tick_params(axis='x', rotation=45)

        # Std Boxplot
        sns.boxplot(x='location_name', y='std', data=category_df, order=locations_order, ax=axes[1], palette='Set2',hue='location_name', legend=False)
        axes[1].set_title(f'Standard Deviation', fontsize=12)
        axes[1].set_xlabel('Region')
        axes[1].set_ylabel('Std Dev')
        axes[1].tick_params(axis='x', rotation=45)

        fig.suptitle(get_fig_title(f'Rating Distribution (Boxplot) for: {category}'), fontsize=16)
        plt.tight_layout()
        plt.show()

        # Violin Plot (Avg, Std)
        fig, axes = plt.subplots(1, 2, figsize=(16, 7))

        # Avg Violin
        sns.violinplot(x='location_name', y='rating_value', data=category_df, order=locations_order, ax=axes[0], palette='Set1', hue='location_name', legend=False)
        axes[0].set_title(f'Average Rating', fontsize=12)
        axes[0].tick_params(axis='x', rotation=45)

        # Std Violin
        sns.violinplot(x='location_name', y='std', data=category_df, order=locations_order, ax=axes[1], palette='Set2', hue='location_name', legend=False)
        axes[1].set_title(f'Standard Deviation', fontsize=12)
        axes[1].tick_params(axis='x', rotation=45)

        fig.suptitle(get_fig_title(f'Rating Distribution (Violin) for: {category}'), fontsize=16)
        plt.tight_layout()
        plt.show()

        # Histogram Comparison (Avg, Std)
        # Per location
        for location in locations_order:
            plot_df = category_df[category_df['location_name'] == location]
            if plot_df.empty: continue

            fig, axes = plt.subplots(1, 2, figsize=(14, 5))

            # Avg Hist
            sns.histplot(data=plot_df, x='rating_value', bins=15, kde=True, ax=axes[0], color='blue')
            axes[0].set_title(f'Average Rating', fontsize=11)
            axes[0].set_xlabel('Rating Value')

            # Std Hist
            sns.histplot(data=plot_df, x='std', bins=15, kde=True, ax=axes[1], color='orange')
            axes[1].set_title(f'Standard Deviation', fontsize=11)
            axes[1].set_xlabel('Std Dev')

            fig.suptitle(get_fig_title(f'Histograms for: {category} in {location}'), fontsize=14)
            plt.tight_layout()
            plt.show()

# Low-Rating Thresholds
print("Low Rating Bar Charts")

# avg
low_avg_crosstab = pd.crosstab(df_filtered['location_type'], df_filtered['low_rating_by_avg'])
print("\n=== Table 1: Low Rating Counts (Average Rating <= 4) ===")
low_avg_table_display = low_avg_crosstab.rename(columns={0: 'High Rating (>4)', 1: 'Low Rating (<=4)'})
print(low_avg_table_display)

low_avg_crosstab.plot(kind='bar', stacked=True, figsize=(8, 6), color=['skyblue', 'salmon'])
plt.title(get_fig_title('Count of Low Ratings (Avg <= 4) by Location Type'))
plt.xlabel('Location Type')
plt.ylabel('Number of Clinics')
plt.legend(title='Avg Rating <= 4', labels=['No', 'Yes'])
plt.tight_layout()
plt.show()

# 25th Percentile
low_p25_crosstab = pd.crosstab(df_filtered['location_type'], df_filtered['low_rating_by_p25'])
print("\n=== Table 2: Low Rating Counts (25th Percentile <= 4) ===")
low_p25_table_display = low_p25_crosstab.rename(columns={0: 'High Rating (>4)', 1: 'Low Rating (<=4)'})
print(low_p25_table_display)

low_p25_crosstab.plot(kind='bar', stacked=True, figsize=(8, 6), color=['lightgreen', 'orange'])
plt.title(get_fig_title('Count of Low Ratings (P25 <= 4) by Location Type'))
plt.xlabel('Location Type')
plt.ylabel('Number of Clinics')
plt.legend(title='25th Percentile <= 4', labels=['No', 'Yes'])
plt.tight_layout()
plt.show()

# Scatter Plots x: Avg; y: P25/P10
print("Scatter Plots x: Avg; y: P25/P10")

unique_cities = df_filtered['location_name'].unique()
n_cities = len(unique_cities)
cols = 3
rows = math.ceil(n_cities / cols)

fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows), sharex=True, sharey=True)
if n_cities > 1:
    axes = axes.flatten()
else:
    axes = [axes]

for i, city in enumerate(unique_cities):
    subset = df_filtered[df_filtered['location_name'] == city]

    # Scatter: X=Avg, Y=P25
    axes[i].scatter(subset['rating_value'], subset['p25'], alpha=0.6, edgecolors='w', color='blue', label='25th Percentile')

    # Scatter: X=Avg, Y=P10
    axes[i].scatter(subset['rating_value'], subset['p10'], alpha=0.6, edgecolors='w', color='orange', label='10th Percentile')

    axes[i].set_title(f"{city} (n={len(subset)})")
    axes[i].set_xlabel("Average Rating")
    axes[i].set_ylabel("Percentile Rating (P25/P10)")
    axes[i].set_xlim(0.5, 5.5)
    axes[i].set_ylim(0.5, 5.5)
    axes[i].grid(True, linestyle='--', alpha=0.5)
    axes[i].legend()

if n_cities > 1:
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

fig.suptitle(get_fig_title("Scatter Plot: Average Rating vs 25th/10th Percentile by City"), fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

# different categories
for category in categories:
        print(f"Processing Scatter Plots for Category: {category}")
        category_df = df_filtered[df_filtered['category'] == category]

        unique_locations = locations_order
        n_locs = len(unique_locations)
        cols = 3
        rows = math.ceil(n_locs / cols)

        fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows), sharex=True, sharey=True)

        # Flatten axes for easy iteration
        if n_locs > 1:
            axes_flat = axes.flatten()
        else:
            axes_flat = [axes]

        has_data_for_scatter = False

        for i, location in enumerate(unique_locations):
            ax = axes_flat[i]
            subset = category_df[category_df['location_name'] == location]

            if not subset.empty:
                has_data_for_scatter = True
                # Scatter: X=Avg, Y=P25
                ax.scatter(subset['rating_value'], subset['p25'], alpha=0.6, edgecolors='w', color='blue', label='P25')
                # Scatter: X=Avg, Y=P10
                ax.scatter(subset['rating_value'], subset['p10'], alpha=0.6, edgecolors='w', color='orange', label='P10')

                ax.set_title(f"{location} (n={len(subset)})")
                ax.grid(True, linestyle='--', alpha=0.5)
            else:
                ax.text(0.5, 0.5, 'No Data', ha='center', va='center', transform=ax.transAxes)

            if i >= (rows - 1) * cols:
                ax.set_xlabel("Avg Rating")
            if i % cols == 0:
                ax.set_ylabel("P25 / P10")

            ax.set_xlim(0.5, 5.5)
            ax.set_ylim(0.5, 5.5)

            # Add legend only to the first subplot to avoid clutter
            if i == 0 and not subset.empty:
                ax.legend(loc='lower right', fontsize='small')

        # Hide any unused subplots
        if n_locs > 1:
            for j in range(i + 1, len(axes_flat)):
                fig.delaxes(axes_flat[j])

        if has_data_for_scatter:
            fig.suptitle(get_fig_title(f'Scatter Plot (Avg vs P25/P10) by Location for: {category}'), fontsize=16)
            plt.tight_layout(rect=[0, 0.03, 1, 0.95])
            plt.show()
        else:
            plt.close(fig)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np
import math

# --- Configuration ---
final_csv_path = "/content/drive/MyDrive/RA/health_care/dentist_detailed_keywords/output/final_merged/merged_union_final.csv"
fig_counter = 1

def get_fig_title(title_text):
    global fig_counter
    title = f"Figure {fig_counter}: {title_text}"
    fig_counter += 1
    return title

try:
    df = pd.read_csv(final_csv_path)
    print("Successfully loaded final_processed.csv")
except FileNotFoundError:
    print(f"Error: The file '{final_csv_path}' was not found.")
    df = pd.DataFrame()

if not df.empty:
    # --- ZIP Mapping ---
    blacksburg_zips = [24060, 24061, 24062, 24063]
    christiansburg_zips = [24068, 24073]
    roanoke_lynchburg_zips = [
        # Roanoke ZIPs
        24001, 24002, 24003, 24004, 24005, 24006, 24007, 24008, 24009, 24010,
        24011, 24012, 24013, 24014, 24015, 24016, 24017, 24018, 24019, 24022,
        # Lynchburg ZIPs
        24501, 24502, 24503, 24504, 24505, 24506, 24513, 24514, 24515
    ]
    dc_zips = [
        20001, 20002, 20003, 20004, 20005, 20006, 20007, 20008, 20009, 20010,
        20011, 20012, 20013, 20015, 20016, 20017, 20018, 20019, 20020, 20022,
        20023, 20024, 20026, 20027, 20029, 20030, 20032, 20033, 20035, 20036,
        20037, 20038, 20039, 20040, 20041, 20042, 20043, 20044, 20045, 20050,
        20051, 20052, 20053, 20055, 20056, 20057, 20058, 20059, 20060, 20061,
        20062, 20063, 20064, 20065, 20066, 20067, 20068, 20069, 20070, 20071,
        20073, 20074, 20075, 20076, 20077, 20078, 20080, 20081, 20082, 20090,
        20091, 20098,
        # NOVA) ZIP
        22301, 22302, 22303, 22304, 22305, 22306, 22307, 22308, 22309, 22310,
        22311, 22312, 22314, 22201, 22202, 22203, 22204, 22205, 22206, 22207,
        22209, 22211, 22213, 22214, 22041, 22042, 22043, 22044, 22046, 22003,
        22031, 22032, 22033, 22101, 22102, 22103, 22106, 22116, 22118, 22180,
        22181, 22182, 22067, 22039
    ]
    ny_zips = [
        10001, 10002, 10003, 10004, 10005, 10006, 10007, 10008, 10009, 10010,
        10011, 10012, 10013, 10014, 10016, 10017, 10018, 10019, 10020, 10021,
        10022, 10023, 10024, 10025, 10026, 10027, 10028, 10029, 10030, 10031,
        10032, 10033, 10034, 10035, 10036, 10037, 10038, 10039, 10040, 10041,
        10043, 10044, 10045, 10046, 10047, 10048, 10055, 10060, 10065, 10069,
        10072, 10079, 10080, 10081, 10082, 10083, 10087, 10090, 10094, 10095,
        10096, 10098, 10099, 10101, 10102, 10103, 10104, 10105, 10106, 10107,
        10108, 10109, 10110, 10111, 10112, 10113, 10114, 10115, 10116, 10117,
        10118, 10119, 10120, 10121, 10122, 10123, 10124, 10125, 10126, 10128,
        10129, 10130, 10131, 10132, 10133, 10138, 10149, 10150, 10151, 10152,
        10153, 10154, 10155, 10156, 10157, 10158, 10159, 10160, 10161, 10162,
        10163, 10164, 10165, 10166, 10167, 10168, 10169, 10170, 10171, 10172,
        10173, 10174, 10175, 10176, 10177, 10178, 10179, 10184, 10185, 10196,
        10197, 10199, 10203, 10211, 10212, 10213, 10249, 10256, 10257, 10258,
        10259, 10260, 10261, 10265, 10268, 10269, 10270, 10271, 10272, 10273,
        10274, 10275, 10276, 10277, 10278, 10279, 10280, 10281, 10282, 10285,
        10286
    ]
    charlotte_zips = [
        28201, 28202, 28203, 28204, 28205, 28206, 28207, 28208, 28209, 28210,
        28211, 28212, 28213, 28214, 28215, 28216, 28217, 28219, 28220, 28221,
        28222, 28223, 28224, 28226, 28227, 28228, 28229, 28230, 28231, 28232,
        28233, 28234, 28235, 28236, 28237, 28241, 28244, 28246, 28247, 28253,
        28254, 28255, 28256, 28258, 28260, 28262, 28265, 28266, 28269, 28270,
        28271, 28272, 28273, 28274, 28275, 28277, 28278, 28280, 28281, 28282,
        28284, 28285, 28287, 28296, 28299
    ]

    def map_zip_to_location(zip_code):
        if pd.isna(zip_code): return 'Unknown'
        try:
            zip_code = int(zip_code)
        except ValueError:
            return 'Unknown'

        if zip_code in blacksburg_zips: return 'Blacksburg'
        elif zip_code in christiansburg_zips: return 'Christiansburg'
        elif zip_code in roanoke_lynchburg_zips: return 'Roanoke/Lynchburg'
        elif zip_code in dc_zips: return 'DC'
        elif zip_code in ny_zips: return 'NY'
        elif zip_code in charlotte_zips: return 'Charlotte'
        return 'Unknown'

    df['location_name'] = df['zip'].apply(map_zip_to_location)
    df_filtered = df[(df['category'] != 'Unknown') & (df['location_name'] != 'Unknown')].copy()

    # Clean star columns
    star_cols = ['rating_1_star', 'rating_2_star', 'rating_3_star', 'rating_4_star', 'rating_5_star']
    for col in star_cols:
        df_filtered[col] = pd.to_numeric(df_filtered[col], errors='coerce').fillna(0).astype(int)

    # --- Calculate ALL Metrics from Star Counts ---
    print("Calculating metrics (Avg, Std, Percentiles) from star counts...")
    def calculate_metrics(row):
        ratings_distribution = []
        ratings_distribution.extend([1] * row['rating_1_star'])
        ratings_distribution.extend([2] * row['rating_2_star'])
        ratings_distribution.extend([3] * row['rating_3_star'])
        ratings_distribution.extend([4] * row['rating_4_star'])
        ratings_distribution.extend([5] * row['rating_5_star'])

        if not ratings_distribution:
            # Return NaN for all 5 metrics
            return pd.Series([np.nan, np.nan, np.nan, np.nan, np.nan])

        # --- NEW: Calculate Average (Mean) ---
        provider_avg = np.mean(ratings_distribution)

        std_dev = np.std(ratings_distribution, ddof=1) if len(ratings_distribution) > 1 else 0
        p10 = np.percentile(ratings_distribution, 10)
        p25 = np.percentile(ratings_distribution, 25)
        p75 = np.percentile(ratings_distribution, 75)
        iqr = p75 - p25

        # Return 5 metrics
        return pd.Series([provider_avg, std_dev, p10, p25, iqr])

    metrics_df = df_filtered.apply(calculate_metrics, axis=1)

    # --- KEY FIX: Assign unique column names to avoid all conflicts ---
    metrics_df.columns = ['provider_avg', 'provider_std', 'provider_p10', 'provider_p25', 'provider_iqr']

    df_filtered = pd.concat([df_filtered, metrics_df], axis=1)

    # --- Data Cleaning (Based on NEWLY calculated values) ---
    # We drop any row that failed calculation (e.g., had 0 stars)
    df_filtered.dropna(subset=['provider_avg', 'provider_std', 'category', 'location_name'], inplace=True)

    urban_locations = ['DC', 'NY', 'Charlotte']
    df_filtered['location_type'] = df_filtered['location_name'].apply(lambda x: 'Urban' if x in urban_locations else 'Country')

    # --- Low-Rating Thresholds (FIXED: Use provider_avg and provider_p25) ---
    df_filtered['low_rating_by_avg'] = np.where(df_filtered['provider_avg'] <= 4, 1, 0)
    df_filtered['low_rating_by_p25'] = np.where(df_filtered['provider_p25'] <= 4, 1, 0)

    print(f"Data ready. {len(df_filtered)} records.")

    categories = sorted(df_filtered['category'].unique())
    locations_order = sorted(df_filtered['location_name'].unique())

    # ==========================================
    # PLOTTING
    # ==========================================

    for category in categories:
        print(f"Processing Category: {category}")
        category_df = df_filtered[df_filtered['category'] == category]

        # --- Boxplot (Avg, Std) (FIXED) ---
        fig, axes = plt.subplots(1, 2, figsize=(16, 7))

        # Avg Boxplot (Use 'provider_avg')
        sns.boxplot(x='location_name', y='provider_avg', data=category_df, order=locations_order, ax=axes[0], palette='Set1', hue='location_name', legend=False)
        axes[0].set_title(f'Average Rating (Calculated)', fontsize=12)
        axes[0].set_xlabel('Region')
        axes[0].set_ylabel('Rating Value')
        axes[0].tick_params(axis='x', rotation=45)

        # Std Boxplot (Use 'provider_std')
        sns.boxplot(x='location_name', y='provider_std', data=category_df, order=locations_order, ax=axes[1], palette='Set2', hue='location_name', legend=False)
        axes[1].set_title(f'Standard Deviation', fontsize=12)
        axes[1].set_xlabel('Region')
        axes[1].set_ylabel('Std Dev')
        axes[1].tick_params(axis='x', rotation=45)

        fig.suptitle(get_fig_title(f'Rating Distribution (Boxplot) for: {category}'), fontsize=16)
        plt.tight_layout()
        plt.show()

        # --- Violin Plot (Avg, Std) (FIXED) ---
        fig, axes = plt.subplots(1, 2, figsize=(16, 7))

        # Avg Violin (Use 'provider_avg')
        sns.violinplot(x='location_name', y='provider_avg', data=category_df, order=locations_order, ax=axes[0], palette='Set1', hue='location_name', legend=False)
        axes[0].set_title(f'Average Rating (Calculated)', fontsize=12)
        axes[0].tick_params(axis='x', rotation=45)

        # Std Violin (Use 'provider_std')
        sns.violinplot(x='location_name', y='provider_std', data=category_df, order=locations_order, ax=axes[1], palette='Set2', hue='location_name', legend=False)
        axes[1].set_title(f'Standard Deviation', fontsize=12)
        axes[1].tick_params(axis='x', rotation=45)

        fig.suptitle(get_fig_title(f'Rating Distribution (Violin) for: {category}'), fontsize=16)
        plt.tight_layout()
        plt.show()

        # --- Histogram Comparison (Avg, Std) (FIXED) ---
        for location in locations_order:
            plot_df = category_df[category_df['location_name'] == location]
            if plot_df.empty: continue

            fig, axes = plt.subplots(1, 2, figsize=(14, 5))

            # Avg Hist (Use 'provider_avg')
            sns.histplot(data=plot_df, x='provider_avg', bins=15, kde=True, ax=axes[0], color='blue')
            axes[0].set_title(f'Average Rating (Calculated)', fontsize=11)
            axes[0].set_xlabel('Rating Value')
            axes[0].set_xlim(0.5, 5.5)

            # Std Hist (Use 'provider_std')
            sns.histplot(data=plot_df, x='provider_std', bins=15, kde=True, ax=axes[1], color='orange')
            axes[1].set_title(f'Standard Deviation', fontsize=11)
            axes[1].set_xlabel('Std Dev')
            axes[1].set_xlim(0, 3.0)

            fig.suptitle(get_fig_title(f'Histograms for: {category} in {location}'), fontsize=14)
            plt.tight_layout()
            plt.show()

    # --- Low-Rating Thresholds (FIXED) ---
    print("Low Rating Bar Charts")

    # avg (Uses 'low_rating_by_avg', which is now based on 'provider_avg')
    low_avg_crosstab = pd.crosstab(df_filtered['location_type'], df_filtered['low_rating_by_avg'])
    print("\n=== Table 1: Low Rating Counts (Calculated Avg <= 4) ===")
    low_avg_table_display = low_avg_crosstab.rename(columns={0: 'High Rating (>4)', 1: 'Low Rating (<=4)'})
    print(low_avg_table_display)

    low_avg_crosstab.plot(kind='bar', stacked=True, figsize=(8, 6), color=['skyblue', 'salmon'])
    plt.title(get_fig_title('Count of Low Ratings (Calculated Avg <= 4) by Location Type'))
    plt.xlabel('Location Type')
    plt.ylabel('Number of Clinics')
    plt.legend(title='Avg Rating <= 4', labels=['No', 'Yes'])
    plt.tight_layout()
    plt.show()

    # 25th Percentile (Uses 'low_rating_by_p25', which is based on 'provider_p25')
    low_p25_crosstab = pd.crosstab(df_filtered['location_type'], df_filtered['low_rating_by_p25'])
    print("\n=== Table 2: Low Rating Counts (25th Percentile <= 4) ===")
    low_p25_table_display = low_p25_crosstab.rename(columns={0: 'High Rating (>4)', 1: 'Low Rating (<=4)'})
    print(low_p25_table_display)

    low_p25_crosstab.plot(kind='bar', stacked=True, figsize=(8, 6), color=['lightgreen', 'orange'])
    plt.title(get_fig_title('Count of Low Ratings (P25 <= 4) by Location Type'))
    plt.xlabel('Location Type')
    plt.ylabel('Number of Clinics')
    plt.legend(title='25th Percentile <= 4', labels=['No', 'Yes'])
    plt.tight_layout()
    plt.show()

    # --- Scatter Plots (FIXED) ---
    # I removed the first, aggregated scatter plot block as it was redundant.
    # This block loops by Category, as you requested.

    print("\n--- Generating Scatter Plots by Category ---")

    for category in categories:
        print(f"Processing Scatter Plots for Category: {category}")
        category_df = df_filtered[df_filtered['category'] == category]

        unique_locations = locations_order
        n_locs = len(unique_locations)
        cols = 3
        rows = math.ceil(n_locs / cols)

        fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows), sharex=True, sharey=True)

        if n_locs > 1:
            axes_flat = axes.flatten()
        else:
            axes_flat = [axes]

        has_data_for_scatter = False

        for i, location in enumerate(unique_locations):
            ax = axes_flat[i]
            subset = category_df[category_df['location_name'] == location]

            if not subset.empty:
                has_data_for_scatter = True
                # Scatter: X=provider_avg, Y=provider_p25
                ax.scatter(subset['provider_avg'], subset['provider_p25'], alpha=0.6, edgecolors='w', color='blue', label='P25')
                # Scatter: X=provider_avg, Y=provider_p10
                ax.scatter(subset['provider_avg'], subset['provider_p10'], alpha=0.6, edgecolors='w', color='orange', label='P10')

                ax.set_title(f"{location} (n={len(subset)})")
                ax.grid(True, linestyle='--', alpha=0.5)
            else:
                ax.text(0.5, 0.5, 'No Data', ha='center', va='center', transform=ax.transAxes)

            if i >= (rows - 1) * cols:
                ax.set_xlabel("Average Rating (Calculated)")
            if i % cols == 0:
                ax.set_ylabel("P25 / P10")

            ax.set_xlim(0.5, 5.5)
            ax.set_ylim(0.5, 5.5)

            if i == 0 and not subset.empty:
                ax.legend(loc='lower right', fontsize='small')

        if n_locs > 1:
            for j in range(i + 1, len(axes_flat)):
                fig.delaxes(axes_flat[j])

        if has_data_for_scatter:
            fig.suptitle(get_fig_title(f'Scatter Plot (Calculated Avg vs P25/P10) by Location for: {category}'), fontsize=16)
            plt.tight_layout(rect=[0, 0.03, 1, 0.95])
            plt.show()
        else:
            plt.close(fig)

    print("Analysis Complete.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np
from IPython.display import display

final_csv_path = "/content/drive/MyDrive/RA/health_care/dentist_detailed_keywords/output/final_merged/merged_union_final.csv"

try:
    df = pd.read_csv(final_csv_path)
    print(f"Successfully loaded '{os.path.basename(final_csv_path)}'")
except FileNotFoundError:
    print(f"Error: The file '{final_csv_path}' was not found.")
    df = pd.DataFrame()

if not df.empty:
    blacksburg_zips = [24060, 24061, 24062, 24063]
    christiansburg_zips = [24068, 24073]
    roanoke_lynchburg_zips = [
        24001, 24002, 24003, 24004, 24005, 24006, 24007, 24008, 24009, 24010,
        24011, 24012, 24013, 24014, 24015, 24016, 24017, 24018, 24019, 24022,
        24501, 24502, 24503, 24504, 24505, 24506, 24513, 24514, 24515
    ]
    dc_zips = [
        20001, 20002, 20003, 20004, 20005, 20006, 20007, 20008, 20009, 20010,
        20011, 20012, 20013, 20015, 20016, 20017, 20018, 20019, 20020, 20022,
        20023, 20024, 20026, 20027, 20029, 20030, 20032, 20033, 20035, 20036,
        20037, 20038, 20039, 20040, 20041, 20042, 20043, 20044, 20045, 20050,
        20051, 20052, 20053, 20055, 20056, 20057, 20058, 20059, 20060, 20061,
        20062, 20063, 20064, 20065, 20066, 20067, 20068, 20069, 20070, 20071,
        20073, 20074, 20075, 20076, 20077, 20078, 20080, 20081, 20082, 20090,
        20091, 20098, 22301, 22302, 22303, 22304, 22305, 22306, 22307, 22308,
        22309, 22310, 22311, 22312, 22314, 22201, 22202, 22203, 22204, 22205,
        22206, 22207, 22209, 22211, 22213, 22214, 22041, 22042, 22043, 22044,
        22046, 22003, 22031, 22032, 22033, 22101, 22102, 22103, 22106, 22116,
        22118, 22180, 22181, 22182, 22067, 22039
    ]
    ny_zips = [
        10001, 10002, 10003, 10004, 10005, 10006, 10007, 10008, 10009, 10010,
        10011, 10012, 10013, 10014, 10016, 10017, 10018, 10019, 10020, 10021,
        10022, 10023, 10024, 10025, 10026, 10027, 10028, 10029, 10030, 10031,
        10032, 10033, 10034, 10035, 10036, 10037, 10038, 10039, 10040, 10041,
        10043, 10044, 10045, 10046, 10047, 10048, 10055, 10060, 10065, 10069,
        10072, 10079, 10080, 10081, 10082, 10083, 10087, 10090, 10094, 10095,
        10096, 10098, 10099, 10101, 10102, 10103, 10104, 10105, 10106, 10107,
        10108, 10109, 10110, 10111, 10112, 10113, 10114, 10115, 10116, 10117,
        10118, 10119, 10120, 10121, 10122, 10123, 10124, 10125, 10126, 10128,
        10129, 10130, 10131, 10132, 10133, 10138, 10149, 10150, 10151, 10152,
        10153, 10154, 10155, 10156, 10157, 10158, 10159, 10160, 10161, 10162,
        10163, 10164, 10165, 10166, 10167, 10168, 10169, 10170, 10171, 10172,
        10173, 10174, 10175, 10176, 10177, 10178, 10179, 10184, 10185, 10196,
        10197, 10199, 10203, 10211, 10212, 10213, 10249, 10256, 10257, 10258,
        10259, 10260, 10261, 10265, 10268, 10269, 10270, 10271, 10272, 10273,
        10274, 10275, 10276, 10277, 10278, 10279, 10280, 10281, 10282, 10285,
        10286
    ]
    charlotte_zips = [
        28201, 28202, 28203, 28204, 28205, 28206, 28207, 28208, 28209, 28210,
        28211, 28212, 28213, 28214, 28215, 28216, 28217, 28219, 28220, 28221,
        28222, 28223, 28224, 28226, 28227, 28228, 28229, 28230, 28231, 28232,
        28233, 28234, 28235, 28236, 28237, 28241, 28244, 28246, 28247, 28253,
        28254, 28255, 28256, 28258, 28260, 28262, 28265, 28266, 28269, 28270,
        28271, 28272, 28273, 28274, 28275, 28277, 28278, 28280, 28281, 28282,
        28284, 28285, 28287, 28296, 28299
    ]

    def map_zip_to_location(zip_code):
        if pd.isna(zip_code): return 'Unknown'
        try:
            zip_code = int(zip_code)
        except ValueError:
            return 'Unknown'
        if zip_code in blacksburg_zips: return 'Blacksburg'
        if zip_code in christiansburg_zips: return 'Christiansburg'
        if zip_code in roanoke_lynchburg_zips: return 'Roanoke/Lynchburg'
        if zip_code in dc_zips: return 'DC'
        if zip_code in ny_zips: return 'NY'
        if zip_code in charlotte_zips: return 'Charlotte'
        return 'Unknown'

    df['location_name'] = df['zip'].apply(map_zip_to_location)

    # clean out discrepancies
    star_cols = ['rating_1_star', 'rating_2_star', 'rating_3_star', 'rating_4_star', 'rating_5_star']
    cols_to_clean = ['votes_count'] + star_cols
    for col in cols_to_clean:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

    df['star_sum'] = df[star_cols].sum(axis=1)
    df['discrepancy_type'] = 0
    df.loc[(df['votes_count'] > 0) & (df['star_sum'] == 0), 'discrepancy_type'] = 1
    df.loc[(df['star_sum'] > 0) & (df['votes_count'] != df['star_sum']), 'discrepancy_type'] = 2

    df_cleaned_for_analysis = df[df['discrepancy_type'] != 2].copy()
    print(f"\nRemoved {len(df) - len(df_cleaned_for_analysis)} records of discrepancy_type 2.")

    print("--- Discrepancy Type Counts ---")
    print(df['discrepancy_type'].value_counts())

    df_filtered = df_cleaned_for_analysis[(df_cleaned_for_analysis['category'] != 'Unknown') & (df_cleaned_for_analysis['location_name'] != 'Unknown')].copy()
    df_filtered['rating_value'] = pd.to_numeric(df_filtered['rating_value'], errors='coerce')
    df_filtered.dropna(subset=['rating_value', 'category', 'location_name'], inplace=True)
    df_filtered['low_rating_dummy'] = np.where(df_filtered['rating_value'] <= 3, 1, 0)
    urban_locations = ['DC', 'NY', 'Charlotte']
    df_filtered['location_type'] = df_filtered['location_name'].apply(lambda x: 'Urban' if x in urban_locations else 'Rural') # Corrected 'Country' to 'Rural' for consistency
    star_cols = ['rating_1_star', 'rating_2_star', 'rating_3_star', 'rating_4_star', 'rating_5_star']
    for col in star_cols:
        df_filtered[col] = pd.to_numeric(df_filtered[col], errors='coerce').fillna(0)
    df_filtered['rating_std'] = df_filtered[star_cols].std(axis=1)

    print(f"After all filtering and preparation, {len(df_filtered)} records remain for analysis.")


    # Boxplot
    categories = sorted(df_filtered['category'].unique())
    locations_order = sorted(df_filtered['location_name'].unique())

    print(f"---Generating boxplots for {len(categories)} categories---")

    for category in categories:
        plt.figure(figsize=(12, 7))
        category_df = df_filtered[df_filtered['category'] == category]

        sns.boxplot(x='location_name', y='rating_value', data=category_df, order=locations_order)

        plt.title(f'Rating Distribution for: {category}', fontsize=16)
        plt.xlabel('Region', fontsize=12)
        plt.ylabel('Rating Value', fontsize=12)
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

    # Histograms each region
    print(f"---Generating Histograms for {len(categories)} categories---")
    for category in categories:
        for location in locations_order:
            plt.figure(figsize=(10, 6))

            plot_df = df_filtered[
                (df_filtered['category'] == category) &
                (df_filtered['location_name'] == location)
            ]

            if plot_df.empty:
                print(f"  Skipping histogram for '{category}' in '{location}' (No data)")
                plt.close()
                continue

            sns.histplot(data=plot_df, x='rating_value', bins=15, kde=True, color='blue')

            plt.title(f'Histogram of Ratings for: {category} in {location}')
            plt.xlabel('Rating Value')
            plt.ylabel('Count of Clinics')
            plt.xlim(1, 5)
            plt.tight_layout()
            plt.show()

    # Violin Plot of Rating Value
    print(f"---Generating Violin Plot for {len(categories)} categories---")
    for category in categories:
        plt.figure(figsize=(12, 7))
        category_df = df_filtered[df_filtered['category'] == category]
        sns.violinplot(x='location_name', y='rating_value', data=category_df, order=locations_order)
        plt.title(f'Rating Distribution for: {category}', fontsize=16)
        plt.xlabel('Region', fontsize=12)
        plt.ylabel('Rating Value', fontsize=12)
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

    # low rate crosstab
    low_rating_crosstab = pd.crosstab(df_filtered['location_type'], df_filtered['low_rating_dummy'])
    print("--- Low Rating (<=3) Analysis: Urban vs. Rural ---")
    display(low_rating_crosstab)

    low_rating_crosstab.plot(kind='bar', stacked=True, figsize=(8, 6), color=['skyblue', 'salmon'])
    plt.title('Count of Low Ratings (<=3) by Location Type')
    plt.xlabel('Location Type')
    plt.ylabel('Number of Clinics')
    plt.xticks(rotation=0)
    plt.legend(title='Low Rating (<=3)', labels=['No (>3)', 'Yes (<=3)'])
    plt.tight_layout()
    plt.show()

    #std
    std_by_category_location_series = df_filtered.groupby(['category', 'location_name'])['rating_std'].mean()
    print(std_by_category_location_series.unstack())
    std_by_category_location = df_filtered.groupby(['category', 'location_name'], as_index=False)['rating_std'].mean()
    print("--- Average Rating Standard Deviation by Category and Location ---")


    if not std_by_category_location.empty:
        for category in categories:
            plt.figure(figsize=(12, 7))

            plot_df = std_by_category_location[std_by_category_location['category'] == category]

            if plot_df.empty:
                print(f"  Skipping std dev plot for '{category}' (No data)")
                plt.close()
                continue

            # Applied fix
            sns.barplot(x='location_name', y='rating_std', data=plot_df, order=locations_order, palette="coolwarm", hue='location_name', legend=False)

            plt.title(f'Average Rating Standard Deviation for: {category}', fontsize=16)
            plt.xlabel('Region', fontsize=12)
            plt.ylabel('Average Standard Deviation of Star Counts', fontsize=12)
            plt.xticks(rotation=45, ha='right')
            plt.tight_layout()
            plt.savefig(f'std_dev_ratings_cat_{category}.png')
            plt.show()
    else:
        print("Could not generate standard deviation data from groupby.")